<a href="https://colab.research.google.com/github/333universe/NPKmathClaud/blob/main/NKPmathClaud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
NKP-Math: claim checking where claims are MATHEMATICAL OBJECTS and sources are COMPUTATIONS.

Text engines can only compare strings; this one asks "is it true?" and computes the answer.
Claim types (each returns a Verdict):
  identity(lhs, rhs)            symbolic proof, else 50-digit random-point testing (finds counterexamples)
  value(expr, "3.14159")        evaluate to 60 digits, compare to the claimed decimal at ITS precision
  exceeds(expr, x, lo, hi, c)   does |f(x)| exceed c somewhere?  interval arithmetic + branch-and-bound:
                                can PROVE "never" (sup bound) or FIND a witness
  dimensions("F = m*a", dims)   units check, with symbolic exponents ("consistent only if b = 0")
  coincidence(a, b, formula)    evidence in BITS = precision bits - formula bits - search bits
Statuses: PROVED, VERIFIED, SUPPORTED, CONSISTENT (accept) | REFUTED, ILL_FORMED, COINCIDENCE (reject)
          CONDITIONAL, UNDETERMINED, SIGNIFICANT (reported as-is)
A SourceLedger keeps the track record of whoever made each claim (NKP's source-reliability idea, but
with ground truth supplied by computation instead of consensus).

Needs sympy and mpmath (both preinstalled in Colab).   python nkp_math.py --test
"""
from __future__ import annotations

import math
import random
import re
import sys
import time
from collections import Counter
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import mpmath as mp
import sympy as sp

mp.mp.dps = 50
ACCEPT = {"PROVED", "VERIFIED", "SUPPORTED", "CONSISTENT", "SIGNIFICANT"}
REJECT = {"REFUTED", "ILL_FORMED", "COINCIDENCE"}

_FUNCS = ("sin", "cos", "tan", "exp", "log", "sqrt", "Abs", "atan", "sinh", "cosh", "tanh")
_GLOBALS = {k: getattr(sp, k) for k in ("Symbol", "Integer", "Float", "Rational", "Function", "Dummy") + _FUNCS}
_GLOBALS["__builtins__"] = {}
_LOCALS = {"phi": sp.GoldenRatio, "pi": sp.pi, "euler": sp.E}
_TRANSFORMS = sp.parsing.sympy_parser.standard_transformations + (sp.parsing.sympy_parser.convert_xor,)


def parse(s: str):
    """Parse text into a sympy expression.  Decimals become exact Rationals ("0.3333" is NOT a binary float)."""
    s = re.sub(r"(?<![\w.])(\d+\.\d+)(?![\w.])", r"Rational('\1')", s)
    return sp.parse_expr(s, local_dict=dict(_LOCALS), global_dict=dict(_GLOBALS), transformations=_TRANSFORMS)


@dataclass
class Verdict:
    status: str
    detail: str
    ms: float = 0.0

    @property
    def accepted(self) -> bool:
        return self.status in ACCEPT


class SourceLedger:
    """Track record of claim sources, scored by what computation found."""

    def __init__(self):
        self.c: Dict[str, Counter] = {}

    def record(self, source: Optional[str], status: str) -> None:
        if source is None:
            return
        k = self.c.setdefault(source, Counter())
        if status in ACCEPT:
            k["true"] += 1
        elif status in REJECT:
            k["false"] += 1
        else:
            k["open"] += 1

    def reliability(self, source: str) -> float:
        k = self.c.get(source, Counter())
        return (k["true"] + 1) / (k["true"] + k["false"] + 2)      # Beta(1,1) prior

    def table(self) -> List[Tuple[str, int, int, int, float]]:
        return sorted(((s, k["true"], k["false"], k["open"], self.reliability(s)) for s, k in self.c.items()),
                      key=lambda r: -r[4])


class NKPMath:
    def __init__(self, n_points: int = 12, seed: int = 0, node_limit: int = 4000):
        self.n_points, self.rng, self.node_limit = n_points, random.Random(seed), node_limit
        self.ledger = SourceLedger()

    def _done(self, v: Verdict, t0: float, source: Optional[str]) -> Verdict:
        v.ms = (time.perf_counter() - t0) * 1000
        self.ledger.record(source, v.status)
        return v

    # ------------------------------------------------------------------ identity
    def identity(self, lhs: str, rhs: str, domain: Optional[Dict[str, Tuple[float, float]]] = None,
                 source: Optional[str] = None) -> Verdict:
        t0 = time.perf_counter()
        L, R = parse(lhs), parse(rhs)
        d = L - R
        syms = sorted(d.free_symbols, key=str)
        if not syms:
            if sp.simplify(d) == 0:
                return self._done(Verdict("PROVED", "exact: the difference simplifies to 0"), t0, source)
            val = sp.N(d, 60)
            if abs(val) > sp.Float(10) ** -40:
                return self._done(Verdict("REFUTED", f"lhs - rhs = {sp.N(d, 12)}"), t0, source)
            return self._done(Verdict("SUPPORTED", "agrees to 40 digits (not proved symbolically)"), t0, source)
        domain = domain or {}
        fL, fR = sp.lambdify(syms, L, modules="mpmath"), sp.lambdify(syms, R, modules="mpmath")
        for _ in range(self.n_points):
            pt = []
            for s in syms:
                lo, hi = domain.get(str(s), (0.1, 3.0))
                pt.append(mp.mpf(lo) + (mp.mpf(hi) - mp.mpf(lo)) * mp.mpf(self.rng.getrandbits(170)) / mp.mpf(2) ** 170)
            a, b = fL(*pt), fR(*pt)
            if abs(a - b) > mp.mpf(10) ** -35 * max(1, abs(a), abs(b)):
                where = ", ".join(f"{s}={mp.nstr(v, 6)}" for s, v in zip(syms, pt))
                return self._done(Verdict("REFUTED", f"counterexample at {where}: lhs-rhs = {mp.nstr(a - b, 6)}"), t0, source)
        if sp.simplify(d) == 0 or sp.simplify(sp.expand_trig(d)) == 0:
            return self._done(Verdict("PROVED", "symbolic: the difference simplifies to 0"), t0, source)
        return self._done(Verdict("SUPPORTED", f"agrees to 35 digits at {self.n_points} random points; not proved"), t0, source)

    # ------------------------------------------------------------------ value
    def value(self, expr: str, claimed: str, source: Optional[str] = None) -> Verdict:
        t0 = time.perf_counter()
        v = mp.mpmathify(str(sp.N(parse(expr), 60)))
        decimals = len(claimed.split(".")[1]) if "." in claimed else 0
        tol = mp.mpf(10) ** (-decimals) / 2
        err = abs(v - mp.mpf(claimed))
        if err <= tol * (1 + mp.mpf(10) ** -15):
            return self._done(Verdict("VERIFIED", f"matches to the claimed {decimals} decimals (true value {mp.nstr(v, decimals + 3)})"), t0, source)
        return self._done(Verdict("REFUTED", f"claimed {claimed}; true value {mp.nstr(v, decimals + 3)} (error {mp.nstr(err, 3)})"), t0, source)

    # ------------------------------------------------------------------ bound (interval arithmetic)
    def exceeds(self, expr: str, var: str, lo, hi, c, source: Optional[str] = None) -> Verdict:
        """Claim: there is an x in [lo, hi] with |f(x)| > c.  Proves 'never' or finds a witness."""
        t0 = time.perf_counter()
        x = sp.Symbol(var)
        e = parse(expr).subs(sp.GoldenRatio, (1 + sp.sqrt(5)) / 2)
        iv = mp.iv
        ns = {n: getattr(iv, n) for n in ("sin", "cos", "tan", "exp", "log", "sqrt") if hasattr(iv, n)}
        ns.update({"Abs": abs, "pi": iv.pi, "e": iv.e, "mpf": iv.mpf})   # constants must be intervals too          # (mpmath intervals have no atan/sinh: such terms are unsupported)
        f_iv = sp.lambdify(x, e, modules=[ns, "mpmath"])
        f_pt = sp.lambdify(x, e, modules="mpmath")
        c = mp.mpf(str(c)) if isinstance(c, str) else mp.mpf(c)
        lo, hi = mp.mpf(lo), mp.mpf(hi)

        def upper(a, b):
            r = f_iv(iv.mpf([a, b]))
            return max(abs(r.a), abs(r.b)) if hasattr(r, "a") else abs(r)

        U = upper(lo, hi)
        if U <= c:
            return self._done(Verdict("REFUTED", f"proved by interval arithmetic: |f| <= {mp.nstr(U, 8)} <= {mp.nstr(c, 8)} on [{lo}, {hi}]"), t0, source)
        stack, nodes, width0 = [(lo, hi)], 0, hi - lo
        while stack and nodes < self.node_limit:
            a, b = stack.pop()
            nodes += 1
            if upper(a, b) <= c:
                continue
            m = (a + b) / 2
            fm = abs(f_pt(m))
            if fm > c:
                return self._done(Verdict("PROVED", f"witness x = {mp.nstr(m, 12)}: |f| = {mp.nstr(fm, 10)} > {mp.nstr(c, 8)}"), t0, source)
            if (b - a) < width0 * mp.mpf(10) ** -14:
                continue
            stack.extend([(a, m), (m, b)])
        if not stack:
            return self._done(Verdict("REFUTED", f"proved by branch-and-bound: |f| <= {mp.nstr(c, 8)} everywhere ({nodes} boxes)"), t0, source)
        return self._done(Verdict("UNDETERMINED", f"no witness and no proof after {nodes} boxes"), t0, source)

    # ------------------------------------------------------------------ dimensions
    def dimensions(self, equation: str, dims: Dict[str, object], source: Optional[str] = None) -> Verdict:
        t0 = time.perf_counter()
        D = {k: self._dimspec(v) for k, v in dims.items()}
        issues: List[str] = []
        constraints: List[sp.Expr] = []

        def sub(a, b):
            return {k: sp.simplify(a.get(k, 0) - b.get(k, 0)) for k in set(a) | set(b)}

        def same(a, b, ctx):
            for k, dv in sub(a, b).items():
                if dv == 0:
                    continue
                if dv.free_symbols:
                    constraints.append(dv)
                else:
                    issues.append(f"{ctx}: [{self._fmt(a)}] vs [{self._fmt(b)}]")
                    return

        def dim(e):
            if e.is_Number or e in (sp.pi, sp.E, sp.GoldenRatio):
                return {}
            if e.is_Symbol:
                return dict(D.get(str(e), {}))
            if e.is_Add:
                first = dim(e.args[0])
                for a in e.args[1:]:
                    same(first, dim(a), f"cannot add terms in {sp.sstr(e)}")
                return first
            if e.is_Mul:
                out: Dict[str, sp.Expr] = {}
                for a in e.args:
                    for k, v in dim(a).items():
                        out[k] = sp.simplify(out.get(k, 0) + v)
                return out
            if e.is_Pow:
                b, ex = e.args
                if dim(ex):
                    issues.append(f"exponent has units in {sp.sstr(e)}")
                bd = dim(b)
                return {k: sp.simplify(v * ex) for k, v in bd.items()} if bd else {}
            if e.func in (sp.sin, sp.cos, sp.tan, sp.exp, sp.log, sp.atan, sp.sinh, sp.cosh, sp.tanh):
                same({}, dim(e.args[0]), f"argument of {e.func.__name__} must be dimensionless, in {sp.sstr(e)}")
                return {}
            if e.func == sp.Abs:
                return dim(e.args[0])
            issues.append(f"unsupported term {sp.sstr(e)}")
            return {}

        lhs, rhs = equation.split("=")
        same(dim(parse(lhs)), dim(parse(rhs)), "both sides of the equation")
        if issues:
            return self._done(Verdict("ILL_FORMED", "; ".join(issues)), t0, source)
        if constraints:
            unknowns = sorted({s for c in constraints for s in c.free_symbols}, key=str)
            sol = sp.solve(constraints, unknowns, dict=True)
            if not sol:
                return self._done(Verdict("ILL_FORMED", f"no choice of {unknowns} makes the units agree"), t0, source)
            return self._done(Verdict("CONDITIONAL", f"units agree ONLY if {sol[0]}"), t0, source)
        return self._done(Verdict("CONSISTENT", "units agree"), t0, source)

    @staticmethod
    def _dimspec(v):
        if isinstance(v, dict):
            return {k: sp.nsimplify(x) for k, x in v.items()}
        out = {}
        for tok in str(v).split():
            m = re.fullmatch(r"([A-Za-z]+)(?:\^(-?[\d/]+))?", tok)
            out[m.group(1)] = sp.Rational(m.group(2)) if m.group(2) else sp.Integer(1)
        return out

    @staticmethod
    def _fmt(d):
        s = " ".join(f"{k}^{v}" for k, v in sorted(d.items()) if v != 0)
        return s or "dimensionless"

    # ------------------------------------------------------------------ coincidence meter
    @staticmethod
    def formula_bits(formula: str, n_leaf: int = 30, n_op: int = 6) -> float:
        e = parse(formula)
        leaves = [a for a in sp.preorder_traversal(e) if a.is_Atom]
        bits = 0.0
        for a in leaves:
            bits += math.log2(n_leaf)
            if a.is_Integer and abs(a) > 12:
                bits += math.log2(abs(int(a)) + 1)          # big integers cost more to specify
        return bits + sp.count_ops(e) * math.log2(n_op)

    def coincidence(self, a: float, b: float, formula: str = "", rel_unc: float = 0.0,
                    search_bits: float = 8.6, threshold: float = 10.0, source: Optional[str] = None) -> Verdict:
        """Is 'a is close to b' evidence of a real relation?  net bits = precision - formula - search."""
        t0 = time.perf_counter()
        rel = max(abs(a - b) / max(abs(a), abs(b)), rel_unc)
        prec = math.log2(1 / (2 * rel)) if rel > 0 else 64.0
        fb = self.formula_bits(formula) if formula else 0.0
        net = prec - fb - search_bits
        txt = f"match to {rel:.2e} = {prec:.1f} bits; formula costs {fb:.1f} bits; search freedom {search_bits:.1f} bits; net {net:+.1f} bits"
        return self._done(Verdict("SIGNIFICANT" if net >= threshold else "COINCIDENCE", txt), t0, source)


# ----------------------------------------------------------------------------- tests
def run_tests() -> None:
    m = NKPMath()
    assert m.identity("sin(x)**2 + cos(x)**2", "1").status == "PROVED"
    r = m.identity("sin(x)", "x - x**3/6")
    assert r.status == "REFUTED" and "counterexample" in r.detail
    assert m.identity("x", "x + 10**(-20)*x**2").status == "REFUTED"                    # a 1e-20 near-identity
    assert m.identity("3*0.3333", "1").status == "REFUTED"                               # 0.9999, exact decimal
    assert m.identity("3*Rational(1,3)", "1").status == "PROVED"
    assert m.identity("2*cos(pi/5)", "phi").status in ("PROVED", "SUPPORTED")
    assert m.identity("1 - 3*(10**4 - 1)/(3*10**4)", "10**(-4)").status == "PROVED"     # the 'flaw' F_n = 10^-n
    assert m.value("pi", "3.14159").status == "VERIFIED" and m.value("pi", "3.14160").status == "REFUTED"
    assert m.value("2*pi/phi**2", "2.3999632297").status == "VERIFIED"
    assert m.value("3**9", "19683").status == "VERIFIED"
    # the 'balanced gate' claim: potential exceeds 1/3 somewhere on [0, 10]
    g = m.exceeds("sin(x*pi)*phi**(x/10)/(pi*phi)", "x", 0, 10, "0.3333333333333333")
    assert g.status == "REFUTED" and "interval" in g.detail, g
    # a narrow peak that a coarse grid would miss
    p = m.exceeds("2*exp(-((x-3.14159)/0.001)**2)", "x", 0, 10, "1.9")
    assert p.status == "PROVED", p
    assert m.exceeds("2*exp(-((x-3.14159)/0.001)**2)", "x", 0, 10, "2.01").status == "REFUTED"
    base = {"m": "M", "a": "L T^-2", "v": "L T^-1", "d": "L", "t": "T", "F": "M L T^-2", "En": "M L^2 T^-2"}
    assert m.dimensions("F = m*a", base).status == "CONSISTENT"
    assert m.dimensions("En = m*v**2/2", base).status == "CONSISTENT"
    bad = m.dimensions("En = m*v", base)
    assert bad.status == "ILL_FORMED", bad
    assert m.dimensions("En = m*v**2 + m*v", base).status == "ILL_FORMED"
    assert m.dimensions("En = m*sin(d)", base).status == "ILL_FORMED"                    # sin of a length
    # Copilot's 'single equation': consistent with GR only if b*s = 0
    gr = {"Gmn": "L^-2", "G": "L^3 M^-1 T^-2", "c": "L T^-1", "Tmn": "M L^-1 T^-2", "X": "M L^-1 T^-2"}
    assert m.dimensions("Gmn = 8*pi*G/c**4*Tmn", gr).status == "CONSISTENT"
    cond = m.dimensions("Gmn = 8*pi*G/c**4*(I*sqrt(X))**(1+b)*Tmn/sqrt(X)", gr)
    assert cond.status == "CONDITIONAL" and "b: 0" in cond.detail, cond
    # coincidence meter: golden angle (deg) vs 1/alpha is a coincidence; a 1e-14 match to a simple formula is not
    ga = 360 * (1 - 1 / ((1 + 5 ** 0.5) / 2))
    assert m.coincidence(ga, 137.035999084, "360*(1-1/phi)").status == "COINCIDENCE"
    assert m.coincidence(3.141592653589793, 3.1415926535897936, "pi").status == "SIGNIFICANT"
    # ledger
    m2 = NKPMath()
    m2.identity("x", "x+1", source="chatA"); m2.identity("x", "x", source="chatA"); m2.identity("x", "x+1", source="chatB")
    assert m2.ledger.reliability("chatA") > m2.ledger.reliability("chatB")
    print("all 20 checks passed")


if __name__ == "__main__":
    run_tests() if "--test" in sys.argv else print(__doc__)


NKP-Math: claim checking where claims are MATHEMATICAL OBJECTS and sources are COMPUTATIONS.

Text engines can only compare strings; this one asks "is it true?" and computes the answer.
Claim types (each returns a Verdict):
  identity(lhs, rhs)            symbolic proof, else 50-digit random-point testing (finds counterexamples)
  value(expr, "3.14159")        evaluate to 60 digits, compare to the claimed decimal at ITS precision
  exceeds(expr, x, lo, hi, c)   does |f(x)| exceed c somewhere?  interval arithmetic + branch-and-bound:
                                can PROVE "never" (sup bound) or FIND a witness
  dimensions("F = m*a", dims)   units check, with symbolic exponents ("consistent only if b = 0")
  coincidence(a, b, formula)    evidence in BITS = precision bits - formula bits - search bits
Statuses: PROVED, VERIFIED, SUPPORTED, CONSISTENT (accept) | REFUTED, ILL_FORMED, COINCIDENCE (reject)
          CONDITIONAL, UNDETERMINED, SIGNIFICANT (reported as-is)
A SourceLedger kee

In [ ]:
%%writefile nkp_math.py
"""
NKP-Math: claim checking where claims are MATHEMATICAL OBJECTS and sources are COMPUTATIONS.

Text engines can only compare strings; this one asks "is it true?" and computes the answer.
Claim types (each returns a Verdict):
  identity(lhs, rhs)            symbolic proof, else 50-digit random-point testing (finds counterexamples)
  value(expr, "3.14159")        evaluate to 60 digits, compare to the claimed decimal at ITS precision
  exceeds(expr, x, lo, hi, c)   does |f(x)| exceed c somewhere?  interval arithmetic + branch-and-bound:
                                can PROVE "never" (sup bound) or FIND a witness
  dimensions("F = m*a", dims)   units check, with symbolic exponents ("consistent only if b = 0")
  coincidence(a, b, formula)    evidence in BITS = precision bits - formula bits - search bits
Statuses: PROVED, VERIFIED, SUPPORTED, CONSISTENT (accept) | REFUTED, ILL_FORMED, COINCIDENCE (reject)
          CONDITIONAL, UNDETERMINED, SIGNIFICANT (reported as-is)
A SourceLedger keeps the track record of whoever made each claim (NKP's source-reliability idea, but
with ground truth supplied by computation instead of consensus).

Needs sympy and mpmath (both preinstalled in Colab).   python nkp_math.py --test
"""
from __future__ import annotations

import math
import random
import re
import sys
import time
from collections import Counter
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import mpmath as mp
import sympy as sp

mp.mp.dps = 50
ACCEPT = {"PROVED", "VERIFIED", "SUPPORTED", "CONSISTENT", "SIGNIFICANT"}
REJECT = {"REFUTED", "ILL_FORMED", "COINCIDENCE"}

_FUNCS = ("sin", "cos", "tan", "exp", "log", "sqrt", "Abs", "atan", "sinh", "cosh", "tanh")
_GLOBALS = {k: getattr(sp, k) for k in ("Symbol", "Integer", "Float", "Rational", "Function", "Dummy") + _FUNCS}
_GLOBALS["__builtins__"] = {}
_LOCALS = {"phi": sp.GoldenRatio, "pi": sp.pi, "euler": sp.E}
_TRANSFORMS = sp.parsing.sympy_parser.standard_transformations + (sp.parsing.sympy_parser.convert_xor,)


def parse(s: str):
    """Parse text into a sympy expression.  Decimals become exact Rationals ("0.3333" is NOT a binary float)."""
    s = re.sub(r"(?<![\w.])(\d+\.\d+)(?![\w.])", r"Rational('\1')", s)
    return sp.parse_expr(s, local_dict=dict(_LOCALS), global_dict=dict(_GLOBALS), transformations=_TRANSFORMS)


@dataclass
class Verdict:
    status: str
    detail: str
    ms: float = 0.0

    @property
    def accepted(self) -> bool:
        return self.status in ACCEPT


class SourceLedger:
    """Track record of claim sources, scored by what computation found."""

    def __init__(self):
        self.c: Dict[str, Counter] = {}

    def record(self, source: Optional[str], status: str) -> None:
        if source is None:
            return
        k = self.c.setdefault(source, Counter())
        if status in ACCEPT:
            k["true"] += 1
        elif status in REJECT:
            k["false"] += 1
        else:
            k["open"] += 1

    def reliability(self, source: str) -> float:
        k = self.c.get(source, Counter())
        return (k["true"] + 1) / (k["true"] + k["false"] + 2)      # Beta(1,1) prior

    def table(self) -> List[Tuple[str, int, int, int, float]]:
        return sorted(((s, k["true"], k["false"], k["open"], self.reliability(s)) for s, k in self.c.items()),
                      key=lambda r: -r[4])


class NKPMath:
    def __init__(self, n_points: int = 12, seed: int = 0, node_limit: int = 4000):
        self.n_points, self.rng, self.node_limit = n_points, random.Random(seed), node_limit
        self.ledger = SourceLedger()

    def _done(self, v: Verdict, t0: float, source: Optional[str]) -> Verdict:
        v.ms = (time.perf_counter() - t0) * 1000
        self.ledger.record(source, v.status)
        return v

    # ------------------------------------------------------------------ identity
    @staticmethod
    def _rational_nonzero(expr, syms) -> bool:
        """True only if expr is PROVABLY a nonzero rational function with rational coefficients."""
        try:
            num, _ = sp.fraction(sp.cancel(expr))
            P = sp.Poly(num, *syms)
            return (not P.is_zero) and all(c.is_Rational for c in P.coeffs())
        except Exception:
            return False

    def identity(self, lhs: str, rhs: str, domain: Optional[Dict[str, Tuple[float, float]]] = None,
                 source: Optional[str] = None) -> Verdict:
        """Is lhs == rhs?  Without a declared domain the claim is read as 'for all real inputs': it is tested on
        positive AND negative values, and a claim true only for positives is reported CONDITIONAL."""
        t0 = time.perf_counter()
        L, R = parse(lhs), parse(rhs)
        d = L - R
        syms = sorted(d.free_symbols, key=str)
        if not syms:
            if sp.simplify(d) == 0:
                return self._done(Verdict("PROVED", "exact: the difference simplifies to 0"), t0, source)
            val = sp.N(d, 60)
            if abs(val) > sp.Float(10) ** -40:
                return self._done(Verdict("REFUTED", f"lhs - rhs = {sp.N(d, 12)}"), t0, source)
            return self._done(Verdict("SUPPORTED", "agrees to 40 digits (not proved symbolically)"), t0, source)
        # exact algebra first: catches differences of ANY size between rational expressions
        try:
            dc = sp.cancel(d)
        except Exception:
            dc = d
        if dc == 0:
            return self._done(Verdict("PROVED", "exact: the rational difference cancels to 0"), t0, source)
        if self._rational_nonzero(d, syms):
            return self._done(Verdict("REFUTED", f"exact: lhs - rhs = {sp.sstr(dc)[:70]}, provably nonzero"), t0, source)
        DIGITS, TOL = 120, mp.mpf(10) ** -90
        with mp.workdps(DIGITS):
            fL, fR = sp.lambdify(syms, L, modules="mpmath"), sp.lambdify(syms, R, modules="mpmath")

            def first_failure(dom, signed):
                for _ in range(self.n_points):
                    pt = []
                    for s_ in syms:
                        lo, hi = dom.get(str(s_), (0.1, 3.0))
                        v = mp.mpf(lo) + (mp.mpf(hi) - mp.mpf(lo)) * mp.mpf(self.rng.getrandbits(420)) / mp.mpf(2) ** 420
                        pt.append(-v if (signed and self.rng.random() < 0.5) else v)
                    a_, b_ = fL(*pt), fR(*pt)
                    if abs(a_ - b_) > TOL * max(1, abs(a_), abs(b_)):
                        where = ", ".join(f"{s_}={mp.nstr(v, 6)}" for s_, v in zip(syms, pt))
                        return f"{where}: lhs-rhs = {mp.nstr(a_ - b_, 6)}"
                return None

            fail = first_failure(domain or {}, signed=False)
            if fail:
                return self._done(Verdict("REFUTED", f"counterexample at {fail}"), t0, source)
            if domain is None:
                fail = first_failure({}, signed=True)
                if fail:
                    return self._done(Verdict("CONDITIONAL", f"holds for positive inputs only; fails at {fail}"), t0, source)
        if sp.simplify(d) == 0 or sp.simplify(sp.expand_trig(d)) == 0:
            return self._done(Verdict("PROVED", "symbolic: the difference simplifies to 0"), t0, source)
        return self._done(Verdict("SUPPORTED", f"agrees to 90 digits at {self.n_points} random points (positive and negative); not proved"), t0, source)

    # ------------------------------------------------------------------ value
    def value(self, expr: str, claimed: str, source: Optional[str] = None) -> Verdict:
        t0 = time.perf_counter()
        decimals = len(claimed.split(".")[1]) if "." in claimed else 0
        with mp.workdps(max(50, decimals + 30)):
            v = mp.mpmathify(str(sp.N(parse(expr), decimals + 40)))
            tol = mp.mpf(10) ** (-decimals) / 2
            err = abs(v - mp.mpf(claimed))
            if err <= tol * (1 + mp.mpf(10) ** -15):
                return self._done(Verdict("VERIFIED", f"matches to the claimed {decimals} decimals (true value {mp.nstr(v, min(decimals + 3, 40))})"), t0, source)
            return self._done(Verdict("REFUTED", f"claimed {claimed[:50]}; true value {mp.nstr(v, min(decimals + 3, 40))} (error {mp.nstr(err, 3)})"), t0, source)

    # ------------------------------------------------------------------ bound (interval arithmetic)
    def exceeds(self, expr: str, var: str, lo, hi, c, source: Optional[str] = None) -> Verdict:
        """Claim: there is an x in [lo, hi] with |f(x)| > c.  Proves 'never' or finds a witness."""
        t0 = time.perf_counter()
        x = sp.Symbol(var)
        e = parse(expr).subs(sp.GoldenRatio, (1 + sp.sqrt(5)) / 2)
        iv = mp.iv
        ns = {n: getattr(iv, n) for n in ("sin", "cos", "tan", "exp", "log", "sqrt") if hasattr(iv, n)}
        ns.update({"Abs": abs, "pi": iv.pi, "e": iv.e, "mpf": iv.mpf})
        f_iv = sp.lambdify(x, e, modules=[ns, "mpmath"])
        f_pt = sp.lambdify(x, e, modules="mpmath")
        c = mp.mpf(str(c)) if isinstance(c, str) else mp.mpf(c)
        lo, hi = mp.mpf(lo), mp.mpf(hi)

        def upper(a, b):
            r = f_iv(iv.mpf([a, b]))
            return max(abs(r.a), abs(r.b)) if hasattr(r, "a") else abs(r)

        U = upper(lo, hi)
        if U <= c:
            return self._done(Verdict("REFUTED", f"proved by interval arithmetic: |f| <= {mp.nstr(U, 8)} <= {mp.nstr(c, 8)} on [{lo}, {hi}]"), t0, source)
        stack, nodes, width0 = [(lo, hi)], 0, hi - lo
        while stack and nodes < self.node_limit:
            a, b = stack.pop()
            nodes += 1
            if upper(a, b) <= c:
                continue
            m = (a + b) / 2
            fm = abs(f_pt(m))
            if fm > c:
                return self._done(Verdict("PROVED", f"witness x = {mp.nstr(m, 12)}: |f| = {mp.nstr(fm, 10)} > {mp.nstr(c, 8)}"), t0, source)
            if (b - a) < width0 * mp.mpf(10) ** -14:
                continue
            stack.extend([(a, m), (m, b)])
        if not stack:
            return self._done(Verdict("REFUTED", f"proved by branch-and-bound: |f| <= {mp.nstr(c, 8)} everywhere ({nodes} boxes)"), t0, source)
        return self._done(Verdict("UNDETERMINED", f"no witness and no proof after {nodes} boxes"), t0, source)

    # ------------------------------------------------------------------ dimensions
    def dimensions(self, equation: str, dims: Dict[str, object], source: Optional[str] = None) -> Verdict:
        t0 = time.perf_counter()
        D = {k: self._dimspec(v) for k, v in dims.items()}
        issues: List[str] = []
        constraints: List[sp.Expr] = []

        def sub(a, b):
            return {k: sp.simplify(a.get(k, 0) - b.get(k, 0)) for k in set(a) | set(b)}

        def same(a, b, ctx):
            for k, dv in sub(a, b).items():
                if dv == 0:
                    continue
                if dv.free_symbols:
                    constraints.append(dv)
                else:
                    issues.append(f"{ctx}: [{self._fmt(a)}] vs [{self._fmt(b)}]")
                    return

        def dim(e):
            if e.is_Number or e in (sp.pi, sp.E, sp.GoldenRatio):
                return {}
            if e.is_Symbol:
                return dict(D.get(str(e), {}))
            if e.is_Add:
                first = dim(e.args[0])
                for a in e.args[1:]:
                    same(first, dim(a), f"cannot add terms in {sp.sstr(e)}")
                return first
            if e.is_Mul:
                out: Dict[str, sp.Expr] = {}
                for a in e.args:
                    for k, v in dim(a).items():
                        out[k] = sp.simplify(out.get(k, 0) + v)
                return out
            if e.is_Pow:
                b, ex = e.args
                if dim(ex):
                    issues.append(f"exponent has units in {sp.sstr(e)}")
                bd = dim(b)
                return {k: sp.simplify(v * ex) for k, v in bd.items()} if bd else {}
            if e.func in (sp.sin, sp.cos, sp.tan, sp.exp, sp.log, sp.atan, sp.sinh, sp.cosh, sp.tanh):
                same({}, dim(e.args[0]), f"argument of {e.func.__name__} must be dimensionless, in {sp.sstr(e)}")
                return {}
            if e.func == sp.Abs:
                return dim(e.args[0])
            issues.append(f"unsupported term {sp.sstr(e)}")
            return {}

        lhs, rhs = equation.split("=")
        same(dim(parse(lhs)), dim(parse(rhs)), "both sides of the equation")
        if issues:
            return self._done(Verdict("ILL_FORMED", "; ".join(issues)), t0, source)
        if constraints:
            unknowns = sorted({s for c in constraints for s in c.free_symbols}, key=str)
            sol = sp.solve(constraints, unknowns, dict=True)
            if not sol:
                return self._done(Verdict("ILL_FORMED", f"no choice of {unknowns} makes the units agree"), t0, source)
            return self._done(Verdict("CONDITIONAL", f"units agree ONLY if {sol[0]}"), t0, source)
        return self._done(Verdict("CONSISTENT", "units agree"), t0, source)

    @staticmethod
    def _dimspec(v):
        if isinstance(v, dict):
            return {k: sp.nsimplify(x) for k, x in v.items()}
        out = {}
        for tok in str(v).split():
            m = re.fullmatch(r"([A-Za-z]+)(?:\^(-?([\d/]+)))?", tok)
            out[m.group(1)] = sp.Rational(m.group(2)) if m.group(2) else sp.Integer(1)
        return out

    @staticmethod
    def _fmt(d):
        s = " ".join(f"{k}^{v}" for k, v in sorted(d.items()) if v != 0)
        return s or "dimensionless"

    # ------------------------------------------------------------------ coincidence meter
    @staticmethod
    def formula_bits(formula: str, n_leaf: int = 30, n_op: int = 6) -> float:
        e = parse(formula)
        leaves = [a for a in sp.preorder_traversal(e) if a.is_Atom]
        bits = 0.0
        for a in leaves:
            bits += math.log2(n_leaf)
            if a.is_Integer and abs(a) > 12:
                bits += math.log2(abs(int(a)) + 1)
        return bits + sp.count_ops(e) * math.log2(n_op)

    def coincidence(self, a: float, b: float, formula: str = "", rel_unc: float = 0.0,
                    search_bits: float = 8.6, threshold: float = 10.0, source: Optional[str] = None) -> Verdict:
        """Is 'a is close to b' evidence of a real relation?  net bits = precision - formula - search."""
        t0 = time.perf_counter()
        rel = max(abs(a - b) / max(abs(a), abs(b)), rel_unc)
        prec = math.log2(1 / (2 * rel)) if rel > 0 else 64.0
        fb = self.formula_bits(formula) if formula else 0.0
        net = prec - fb - search_bits
        txt = f"match to {rel:.2e} = {prec:.1f} bits; formula costs {fb:.1f} bits; search freedom {search_bits:.1f} bits; net {net:+.1f} bits"
        return self._done(Verdict("SIGNIFICANT" if net >= threshold else "COINCIDENCE", txt), t0, source)


# ----------------------------------------------------------------------------- tests
def run_tests() -> None:
    m = NKPMath()
    assert m.identity("sin(x)**2 + cos(x)**2", "1").status == "PROVED"
    r = m.identity("sin(x)", "x - x**3/6")
    assert r.status == "REFUTED" and "counterexample" in r.detail
    assert m.identity("x", "x + 10**(-20)*x**2").status == "REFUTED"
    assert m.identity("3*0.3333", "1").status == "REFUTED"
    assert m.identity("3*Rational(1,3)", "1").status == "PROVED"
    assert m.identity("2*cos(pi/5)", "phi").status in ("PROVED", "SUPPORTED")
    assert m.identity("1 - 3*(10**4 - 1)/(3*10**4)", "10**(-4)").status == "PROVED"
    assert m.value("pi", "3.14159").status == "VERIFIED" and m.value("pi", "3.14160").status == "REFUTED"
    assert m.value("2*pi/phi**2", "2.3999632297").status == "VERIFIED"
    assert m.value("3**9", "19683").status == "VERIFIED"
    g = m.exceeds("sin(x*pi)*phi**(x/10)/(pi*phi)", "x", 0, 10, "0.3333333333333333")
    assert g.status == "REFUTED" and "interval" in g.detail, g
    p = m.exceeds("2*exp(-((x-3.14159)/0.001)**2)", "x", 0, 10, "1.9")
    assert p.status == "PROVED", p
    assert m.exceeds("2*exp(-((x-3.14159)/0.001)**2)", "x", 0, 10, "2.01").status == "REFUTED"
    base = {"m": "M", "a": "L T^-2", "v": "L T^-1", "d": "L", "t": "T", "F": "M L T^-2", "En": "M L^2 T^-2"}
    assert m.dimensions("F = m*a", base).status == "CONSISTENT"
    assert m.dimensions("En = m*v**2/2", base).status == "CONSISTENT"
    bad = m.dimensions("En = m*v", base)
    assert bad.status == "ILL_FORMED", bad
    assert m.dimensions("En = m*v**2 + m*v", base).status == "ILL_FORMED"
    assert m.dimensions("En = m*sin(d)", base).status == "ILL_FORMED"
    gr = {"Gmn": "L^-2", "G": "L^3 M^-1 T^-2", "c": "L T^-1", "Tmn": "M L^-1 T^-2", "X": "M L^-1 T^-2"}
    assert m.dimensions("Gmn = 8*pi*G/c**4*Tmn", gr).status == "CONSISTENT"
    cond = m.dimensions("Gmn = 8*pi*G/c**4*(I*sqrt(X))**(1+b)*Tmn/sqrt(X)", gr)
    assert cond.status == "CONDITIONAL" and "b: 0" in cond.detail, cond
    ga = 360 * (1 - 1 / ((1 + 5 ** 0.5) / 2))
    assert m.coincidence(ga, 137.035999084, "360*(1-1/phi)").status == "COINCIDENCE"
    assert m.coincidence(3.141592653589793, 3.1415926535897936, "pi").status == "SIGNIFICANT"
    m2 = NKPMath()
    m2.identity("x", "x+1", source="chatA"); m2.identity("x", "x", source="chatA"); m2.identity("x", "x+1", source="chatB")
    assert m2.ledger.reliability("chatA") > m2.ledger.reliability("chatB")
    assert m.identity("sqrt(x**2)", "x").status == "CONDITIONAL"
    assert m.identity("sqrt(x**2)", "x", domain={"x": (0.1, 3.0)}).status in ("PROVED", "SUPPORTED")
    assert m.identity("sqrt(x**4)", "x**2").accepted and m.identity("sqrt(x**2)", "Abs(x)").accepted
    assert m.identity("x", "x + 10**(-60)*x**2").status == "REFUTED"
    assert m.identity("y", "y + 10**(-70)*y**3").status == "REFUTED"
    assert m.identity("exp(x)", "exp(x) + 10**(-100)*sin(x)").status == "SUPPORTED"
    from decimal import Decimal
    with mp.workdps(120):
        pi55 = format(Decimal(mp.nstr(mp.pi, 100)).quantize(Decimal(1).scaleb(-55)), "f")
    assert m.value("pi", pi55).status == "VERIFIED"
    assert m.value("pi", pi55[:-1] + str((int(pi55[-1]) + 1) % 10)).status == "REFUTED"
    print("all 28 checks passed")


if __name__ == "__main__":
    run_tests() if "--test" in sys.argv else print(__doc__)

Writing nkp_math.py


In [ ]:

"""
Benchmark for NKP-Math.  Two parts, with different meanings:

PART 1  Synthetic claim zoo (labels known BY CONSTRUCTION; each seed = 130 claims)
   identities (true templates; false by coefficient/trig swaps, tiny 1e-2..1e-25 perturbations,
   and decimal "shortfall" arithmetic), decimal value claims (digit-level errors),
   existence bounds (narrow Gaussian peaks), and unit-consistency equations.
   Baselines: Trust (accept all) | Lookup (text match against a textbook list) |
              Float1 (one random point, float64, tol 1e-9) | Float5 (five points).
   This part measures CAPABILITY COVERAGE: which failure modes does each approach catch?  It is not a
   contest on a shared hard task, and the generator and engine were written by the same person.

PART 2  Real claims taken from the user's own documents (ground truth from independent calculations
   done earlier in the project).  This is the meaningful part.

PRE-REGISTERED CRITERIA (written before any run):
  M1  NKP-Math false-accept rate <= 1% overall
  M2  overall accuracy beats the best baseline by > 2 SE (paired over seeds)
  M3  in no family is NKP-Math worse than the best baseline by > 2 SE
  M4  every real case receives the expected verdict
Dev seeds 0-9; final test = fresh seeds 100-119, run once.

    python nkp_math_bench.py --real
    python nkp_math_bench.py --seeds 0:10 --out dev.json ;  python nkp_math_bench.py --report dev.json
"""
from __future__ import annotations

import json
import math
import random
import re
import statistics
import sys
from decimal import ROUND_HALF_UP, Decimal, getcontext

import mpmath as mp
import numpy as np
import sympy as sp

from nkp_math import NKPMath, parse

getcontext().prec = 90
FAMILIES = ["identity", "shortfall", "value", "bound", "dimensions"]
METHODS = ["Trust", "Lookup", "Float1", "Float5", "NKP-Math"]


def fib(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a


# ------------------------------------------------------------------ generators
def _true_identity(rng):
    k = rng.randrange(9); a = rng.randint(2, 6)
    return [
        (f"sin({a}*x)**2+cos({a}*x)**2", "1"),
        (f"(x+{a})**3", f"x**3+{3*a}*x**2+{3*a*a}*x+{a**3}"),
        (f"x**2-{a*a}", f"(x-{a})*(x+{a})"),
        (f"cos({2*a}*x)", f"1-2*sin({a}*x)**2"),
        ("sin(x+y)", "sin(x)*cos(y)+cos(x)*sin(y)"),
        (f"exp({a}*x+y)", f"exp(x)**{a}*exp(y)"),
        (f"log(x**{a}*y)", f"{a}*log(x)+log(y)"),
        (f"x**{a}-1", "(x-1)*(" + "+".join(f"x**{j}" for j in range(a)) + ")"),
        (f"phi**{a+2}", f"{fib(a+2)}*phi+{fib(a+1)}"),
    ][k]


def _oracle_identity(L, R):
    """Ground truth for mutated identities, computed independently of the engine (80 digits, 24 points)."""
    r = random.Random(7)
    d = parse(L) - parse(R)
    syms = sorted(d.free_symbols, key=str)
    f = sp.lambdify(syms, d, modules="mpmath")
    mp.mp.dps = 80
    try:
        for _ in range(24):
            pt = [mp.mpf(0.1) + mp.mpf(2.9) * mp.mpf(r.getrandbits(250)) / mp.mpf(2) ** 250 for _ in syms]
            if abs(f(*pt)) > mp.mpf(10) ** -60:
                return False
    finally:
        mp.mp.dps = 50
    return True


def gen_identity(rng):
    L, R = _true_identity(rng)
    if rng.random() < 0.5:
        return dict(family="identity", L=L, R=R, label=True)
    mode = rng.choice(["perturb", "perturb", "coef", "swap"])
    if mode == "perturb":
        R2 = f"({R})+10**(-{rng.choice([2, 4, 6, 9, 12, 16, 20, 25])})*x**2"
    elif mode == "coef":
        nums = [m for m in re.finditer(r"(?<![\w.*^])\d+(?![\w.])", R)] or [m for m in re.finditer(r"\d+", R)]
        if nums:
            m = rng.choice(nums); R2 = R[:m.start()] + str(int(m.group()) + 1) + R[m.end():]
        else:
            R2 = f"({R})+x"                                           # no numerals to alter: add a term instead
    else:
        R2 = R.replace("sin", "cos", 1) if "sin" in R else R.replace("exp", "exp(1)*exp", 1) if "exp" in R else f"({R})*2"
    return dict(family="identity", L=L, R=R2, label=_oracle_identity(L, R2))


def gen_shortfall(rng):
    n = rng.randint(3, 25)
    if rng.random() < 0.5:
        return dict(family="shortfall", L=f"3*0.{'3'*n}", R="1", label=False)                     # 3 x 0.33..3 = 1 ?
    return dict(family="shortfall", L=f"3*(10**{n}-1)/(3*10**{n})+10**(-{n})", R="1", label=True)


_VALS = ["pi", "phi", "sqrt(2)", "sqrt(3)", "log(2)", "exp(pi)", "2*pi/phi**2", "pi**2/6", "sqrt(5)", "exp(1)"]


def gen_value(rng):
    e = rng.choice(_VALS); k = rng.randint(3, 28)
    v = Decimal(str(sp.N(parse(e), 80)))
    q = v.quantize(Decimal(1).scaleb(-k), rounding=ROUND_HALF_UP)
    if rng.random() < 0.5:
        return dict(family="value", expr=e, claimed=format(q, "f"), label=True)
    bad = q + rng.choice([-1, 1]) * Decimal(1).scaleb(-k)
    return dict(family="value", expr=e, claimed=format(bad, "f"), label=False)


def gen_bound(rng):
    a = round(rng.uniform(0.5, 5), 3); x0 = round(rng.uniform(1, 9), 4)
    w = rng.choice([0.002, 0.01, 0.05]); kk = rng.choice([1, 3]); eps = rng.choice([0.001, 0.01, 0.05])
    true = rng.random() < 0.5
    c = a * (1 - eps) if true else a * (1 + eps)
    return dict(family="bound", expr=f"{a}*exp(-((x-{x0})/{w})**2)*cos({kk}*(x-{x0}))", c=format(Decimal(c).quantize(Decimal("1e-9")), "f"), label=true)


_QD = {"m": "M", "a": "L T^-2", "v": "L T^-1", "d": "L", "t": "T", "F": "M L T^-2", "En": "M L^2 T^-2", "p": "M L T^-1", "P": "M L^2 T^-3"}
_TRUE_EQ = ["F = m*a", "En = m*v**2/2", "p = m*v", "v = d/t", "P = F*v", "En = F*d", "a = v/t", "En = P*t", "d = v*t", "F = p/t"]
_BAD_EQ = ["En = m*v", "F = m*v", "p = m*v**2", "v = d*t", "P = F*d", "En = m*v**2 + m*v", "a = v*t", "En = F*t", "d = v/t", "F = m*sin(d)", "En = m*exp(t)", "P = F*v + m*a"]


def gen_dims(rng):
    true = rng.random() < 0.5
    eq = rng.choice(_TRUE_EQ if true else _BAD_EQ)
    if rng.random() < 0.5:                                        # harmless decoration: a dimensionless factor
        lhs, rhs = eq.split("="); eq = f" {lhs.strip()} = {rng.choice(['2', '3', 'pi', '1/2'])}*({rhs.strip()})"
        if not true and "sin" in eq or not true and "exp" in eq: pass
    return dict(family="dimensions", eq=eq, label=true)


def make_seed(seed):
    rng = random.Random(seed)
    out = [gen_identity(rng) for _ in range(40)] + [gen_shortfall(rng) for _ in range(10)] + [gen_value(rng) for _ in range(30)]
    out += [gen_bound(rng) for _ in range(20)] + [gen_dims(rng) for _ in range(30)]
    return out


# ------------------------------------------------------------------ baselines and engine
def fparse(s):
    return sp.parse_expr(s.replace("^", "**"), local_dict={"phi": sp.GoldenRatio, "pi": sp.pi, "euler": sp.E})


_LOOKUP = {("sin(2*x)**2+cos(2*x)**2", "1"), ("sin(x+y)", "sin(x)*cos(y)+cos(x)*sin(y)"), ("x**2-4", "(x-2)*(x+2)"),
           ("exp(2*x+y)", "exp(x)**2*exp(y)"), ("phi**4", "3*phi+2"), ("(x+2)**3", "x**3+6*x**2+12*x+8")}
_LOOKUP_VALUES = {("pi", "3.14159265358979"), ("phi", "1.6180339887"), ("sqrt(2)", "1.41421356237"), ("exp(1)", "2.71828182845")}
_LOOKUP_EQ = {"F = m*a", "En = m*v**2/2", "p = m*v", "v = d/t"}


def float_check(c, npts, rng):
    f = c["family"]
    if f in ("identity", "shortfall"):
        d = fparse(c["L"]) - fparse(c["R"])
        syms = sorted(d.free_symbols, key=str)
        fn = sp.lambdify(syms, d, "numpy")
        fl = sp.lambdify(syms, fparse(c["L"]), "numpy")
        for _ in range(npts):
            pt = [rng.uniform(0.1, 3.0) for _ in syms]
            if abs(float(fn(*pt))) > 1e-9 * max(1.0, abs(float(fl(*pt)))):
                return False
        return True
    if f == "value":
        return abs(float(sp.N(fparse(c["expr"]), 15)) - float(c["claimed"])) <= 1e-9
    if f == "bound":
        fn = sp.lambdify(sp.Symbol("x"), fparse(c["expr"]), "numpy")
        xs = np.linspace(0, 10, 400 if npts > 1 else 50)
        return bool(np.max(np.abs(fn(xs))) > float(c["c"]))
    return False                                                  # numeric checks cannot see units


def decide(method, c, eng, rng):
    f = c["family"]
    if method == "Trust":
        return True
    if method == "Lookup":
        if f in ("identity", "shortfall"): return (c["L"], c["R"]) in _LOOKUP
        if f == "value": return (c["expr"], c["claimed"]) in _LOOKUP_VALUES
        if f == "dimensions": return c["eq"] in _LOOKUP_EQ
        return False
    if method == "Float1": return float_check(c, 1, rng)
    if method == "Float5": return float_check(c, 5, rng)
    if f in ("identity", "shortfall"): return eng.identity(c["L"], c["R"]).accepted
    if f == "value": return eng.value(c["expr"], c["claimed"]).accepted
    if f == "bound": return eng.exceeds(c["expr"], "x", 0, 10, c["c"]).status == "PROVED"
    return eng.dimensions(c["eq"], _QD).accepted


def run_seed(seed):
    claims = make_seed(seed); eng = NKPMath(seed=seed); rng = random.Random(seed + 1)
    tally = {m: {fam: [0, 0, 0, 0] for fam in FAMILIES} for m in METHODS}            # correct, total, false-accepts, false-total
    for c in claims:
        for m in METHODS:
            d = decide(m, c, eng, rng)
            t = tally[m][c["family"]]
            t[0] += int(d == c["label"]); t[1] += 1
            if not c["label"]:
                t[3] += 1; t[2] += int(d)
    return tally


# ------------------------------------------------------------------ part 2: real claims from the user's documents
REAL = [
    # (kind, args, expected, source)
    ("identity", ("2*cos(pi/5)", "phi"), "accept", "screenshot:gemini-identity"),
    ("identity", ("phi", "1+1/phi"), "accept", "screenshot:gemini-identity"),
    ("identity", ("phi**2", "phi+1"), "accept", "doc:copilot-parameters"),
    ("identity", ("3*0.3333", "1"), "reject", "screenshot:triadic-flaw"),
    ("identity", ("0.3333+0.3333+0.3333", "0.9999"), "accept", "user:own-arithmetic"),
    ("identity", ("0.3333+0.3333+0.3333+0.0001", "1"), "accept", "user:own-arithmetic"),
    ("identity", ("1-3*(10**8-1)/(3*10**8)", "10**(-8)"), "accept", "screenshot:triadic-flaw"),
    ("value", ("2*pi/phi**2", "2.3999632297"), "accept", "doc:sieve-script"),
    ("value", ("3**9", "19683"), "accept", "screenshot:gemini-tictactoe"),
    ("exceeds", ("sin(x*pi)*phi**(x/10)/(pi*phi)", "x", 0, 10, "0.3333333333333333"), "reject", "doc:gate-script"),
    ("dimensions", ("Gmn = 8*pi*G/c**4*Tmn", {"Gmn": "L^-2", "G": "L^3 M^-1 T^-2", "c": "L T^-1", "Tmn": "M L^-1 T^-2"}), "accept", "gr-baseline"),
    ("dimensions", ("Gmn = 8*pi*G/c**4*(I*sqrt(X))**(1+b)*Tmn/sqrt(X)", {"Gmn": "L^-2", "G": "L^3 M^-1 T^-2", "c": "L T^-1", "Tmn": "M L^-1 T^-2", "X": "M L^-1 T^-2"}), "conditional", "screenshot:copilot-single-equation"),
    ("dimensions", ("Pw = al*Cc*f*V**2", {"Pw": "M L^2 T^-3", "Cc": "Q^2 T^2 M^-1 L^-2", "f": "T^-1", "V": "M L^2 T^-2 Q^-1"}), "accept", "screenshot:google-ai-cpu"),
    ("dimensions", ("Pj = Ic**2*Rr", {"Pj": "M L^2 T^-3", "Ic": "Q T^-1", "Rr": "M L^2 T^-1 Q^-2"}), "accept", "screenshot:google-ai-cpu"),
    ("dimensions", ("Ic = Isat*exp(V)", {"Ic": "Q T^-1", "Isat": "Q T^-1", "V": "M L^2 T^-2 Q^-1"}), "reject", "screenshot:google-ai-cpu"),
    ("dimensions", ("Gmn = rho", {"Gmn": "L^-2"}), "reject", "doc:unified-equation"),
    ("coincidence", (360 * (1 - 1 / ((1 + 5 ** 0.5) / 2)), 137.035999084, "360*(1-1/phi)"), "reject", "user:hydrogen-similarity"),
]


def run_real():
    eng = NKPMath(seed=0); ok = 0
    print(f"{'#':>2} {'source':<34}{'verdict':<13}{'expected':<12} claim / detail")
    for i, (kind, args, exp, src) in enumerate(REAL, 1):
        v = getattr(eng, kind)(*args, source=src)
        got = "accept" if v.accepted else "reject" if v.status in ("REFUTED", "ILL_FORMED", "COINCIDENCE") else "conditional"
        good = got == exp; ok += good
        name = args[0] if isinstance(args[0], str) else f"{args[0]:.4f} vs {args[1]}"
        print(f"{i:>2} {src:<34}{v.status:<13}{exp:<12} {'OK ' if good else 'BAD'} {str(name)[:46]} | {v.detail[:90]}")
    print(f"\nM4: {ok} of {len(REAL)} real claims received the expected verdict -> {'PASS' if ok == len(REAL) else 'FAIL'}")
    print("\nsource track record (true / false / open -> reliability):")
    for s, t, f, o, r in eng.ledger.table():
        print(f"  {s:<34}{t} / {f} / {o}  -> {r:.2f}")
    return ok == len(REAL)


# ------------------------------------------------------------------ reporting
def se(x):
    return statistics.stdev(x) / math.sqrt(len(x)) if len(x) > 1 else 0.0


def report(files):
    runs = []
    for fn in files: runs += json.load(open(fn))
    n = len(runs)

    def acc(r, m, fams):
        c = sum(r[m][f][0] for f in fams); t = sum(r[m][f][1] for f in fams); return c / t

    def fa(r, m):
        a = sum(r[m][f][2] for f in FAMILIES); b = sum(r[m][f][3] for f in FAMILIES); return a / b
    print(f"{n} seeds, {n*130} claims.   accuracy by family (mean over seeds)")
    print(f"{'family':<12}" + "".join(f"{m:>11}" for m in METHODS))
    for fam in FAMILIES + ["ALL"]:
        fams = FAMILIES if fam == "ALL" else [fam]
        print(f"{fam:<12}" + "".join(f"{statistics.mean(acc(r, m, fams) for r in runs):>11.3f}" for m in METHODS))
    print(f"{'false-accept':<12}" + "".join(f"{statistics.mean(fa(r, m) for r in runs):>11.3f}" for m in METHODS))
    best = lambda fams: max(METHODS[:-1], key=lambda m: statistics.mean(acc(r, m, fams) for r in runs))
    print("\nPRE-REGISTERED CRITERIA")
    m1 = statistics.mean(fa(r, "NKP-Math") for r in runs)
    print(f"  M1 false-accept rate {m1:.3f}  {'PASS' if m1 <= 0.01 else 'FAIL'}")
    b = best(FAMILIES); d = [acc(r, "NKP-Math", FAMILIES) - acc(r, b, FAMILIES) for r in runs]
    print(f"  M2 overall vs best baseline ({b}): {statistics.mean(d):+.3f}±{se(d):.3f}  {'PASS' if statistics.mean(d) - 2 * se(d) > 0 else 'FAIL'}")
    ok3 = True
    for fam in FAMILIES:
        b = best([fam]); d = [acc(r, "NKP-Math", [fam]) - acc(r, b, [fam]) for r in runs]
        good = statistics.mean(d) + 2 * se(d) >= 0; ok3 &= good
        print(f"  M3 {fam:<11} vs best baseline ({b}): {statistics.mean(d):+.3f}±{se(d):.3f}  {'ok' if good else 'WORSE'}")
    print(f"  M3 overall: {'PASS' if ok3 else 'FAIL'}")


if __name__ == "__main__":
    a = sys.argv
    if "--real" in a:
        run_real()
    elif "--report" in a:
        report(a[a.index("--report") + 1:])
    elif "--seeds" in a:
        lo, hi = map(int, a[a.index("--seeds") + 1].split(":"))
        res = [run_seed(s) for s in range(lo, hi)]
        json.dump(res, open(a[a.index("--out") + 1], "w"))
        print(f"seeds {lo}:{hi} done")
    else:
        # Default fallback to run the real benchmarks if no cli arguments are passed inside the notebook environment
        print("No CLI arguments matching benchmark flags detected. Running --real validation instead:\n")
        run_real()

No CLI arguments matching benchmark flags detected. Running --real validation instead:

 # source                            verdict      expected     claim / detail
 1 screenshot:gemini-identity        PROVED       accept       OK  2*cos(pi/5) | exact: the difference simplifies to 0
 2 screenshot:gemini-identity        PROVED       accept       OK  phi | exact: the difference simplifies to 0
 3 doc:copilot-parameters            PROVED       accept       OK  phi**2 | exact: the difference simplifies to 0
 4 screenshot:triadic-flaw           REFUTED      reject       OK  3*0.3333 | lhs - rhs = -0.000100000000000
 5 user:own-arithmetic               PROVED       accept       OK  0.3333+0.3333+0.3333 | exact: the difference simplifies to 0
 6 user:own-arithmetic               PROVED       accept       OK  0.3333+0.3333+0.3333+0.0001 | exact: the difference simplifies to 0
 7 screenshot:triadic-flaw           PROVED       accept       OK  1-3*(10**8-1)/(3*10**8) | exact: the difference sim

In [ ]:

"""
Benchmark for NKP-Math.  Two parts, with different meanings:

PART 1  Synthetic claim zoo (labels known BY CONSTRUCTION; each seed = 130 claims)
   identities (true templates; false by coefficient/trig swaps, tiny 1e-2..1e-25 perturbations,
   and decimal "shortfall" arithmetic), decimal value claims (digit-level errors),
   existence bounds (narrow Gaussian peaks), and unit-consistency equations.
   Baselines: Trust (accept all) | Lookup (text match against a textbook list) |
              Float1 (one random point, float64, tol 1e-9) | Float5 (five points).
   This part measures CAPABILITY COVERAGE: which failure modes does each approach catch?  It is not a
   contest on a shared hard task, and the generator and engine were written by the same person.

PART 2  Real claims taken from the user's own documents (ground truth from independent calculations
   done earlier in the project).  This is the meaningful part.

PRE-REGISTERED CRITERIA (written before any run):
  M1  NKP-Math false-accept rate <= 1% overall
  M2  overall accuracy beats the best baseline by > 2 SE (paired over seeds)
  M3  in no family is NKP-Math worse than the best baseline by > 2 SE
  M4  every real case receives the expected verdict
Dev seeds 0-9; final test = fresh seeds 100-119, run once.

    python nkp_math_bench.py --real
    python nkp_math_bench.py --seeds 0:10 --out dev.json ;  python nkp_math_bench.py --report dev.json
"""
from __future__ import annotations

import json
import math
import random
import re
import statistics
import sys
from decimal import ROUND_HALF_UP, Decimal, getcontext

import mpmath as mp
import numpy as np
import sympy as sp

from nkp_math import NKPMath, parse

getcontext().prec = 90
FAMILIES = ["identity", "shortfall", "value", "bound", "dimensions"]
METHODS = ["Trust", "Lookup", "Float1", "Float5", "NKP-Math"]


def fib(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a


# ------------------------------------------------------------------ generators
def _true_identity(rng):
    k = rng.randrange(9); a = rng.randint(2, 6)
    return [
        (f"sin({a}*x)**2+cos({a}*x)**2", "1"),
        (f"(x+{a})**3", f"x**3+{3*a}*x**2+{3*a*a}*x+{a**3}"),
        (f"x**2-{a*a}", f"(x-{a})*(x+{a})"),
        (f"cos({2*a}*x)", f"1-2*sin({a}*x)**2"),
        ("sin(x+y)", "sin(x)*cos(y)+cos(x)*sin(y)"),
        (f"exp({a}*x+y)", f"exp(x)**{a}*exp(y)"),
        (f"log(x**{a}*y)", f"{a}*log(x)+log(y)"),
        (f"x**{a}-1", "(x-1)*(" + "+".join(f"x**{j}" for j in range(a)) + ")"),
        (f"phi**{a+2}", f"{fib(a+2)}*phi+{fib(a+1)}"),
    ][k]


def _oracle_identity(L, R):
    """Ground truth for mutated identities, computed independently of the engine (80 digits, 24 points)."""
    r = random.Random(7)
    d = parse(L) - parse(R)
    syms = sorted(d.free_symbols, key=str)
    f = sp.lambdify(syms, d, modules="mpmath")
    mp.mp.dps = 80
    try:
        for _ in range(24):
            pt = [mp.mpf(0.1) + mp.mpf(2.9) * mp.mpf(r.getrandbits(250)) / mp.mpf(2) ** 250 for _ in syms]
            if abs(f(*pt)) > mp.mpf(10) ** -60:
                return False
    finally:
        mp.mp.dps = 50
    return True


def gen_identity(rng):
    L, R = _true_identity(rng)
    if rng.random() < 0.5:
        return dict(family="identity", L=L, R=R, label=True)
    mode = rng.choice(["perturb", "perturb", "coef", "swap"])
    if mode == "perturb":
        R2 = f"({R})+10**(-{rng.choice([2, 4, 6, 9, 12, 16, 20, 25])})*x**2"
    elif mode == "coef":
        nums = [m for m in re.finditer(r"(?<![\w.*^])\d+(?![\w.])", R)] or [m for m in re.finditer(r"\d+", R)]
        if nums:
            m = rng.choice(nums); R2 = R[:m.start()] + str(int(m.group()) + 1) + R[m.end():]
        else:
            R2 = f"({R})+x"                                           # no numerals to alter: add a term instead
    else:
        R2 = R.replace("sin", "cos", 1) if "sin" in R else R.replace("exp", "exp(1)*exp", 1) if "exp" in R else f"({R})*2"
    return dict(family="identity", L=L, R=R2, label=_oracle_identity(L, R2))


def gen_shortfall(rng):
    n = rng.randint(3, 25)
    if rng.random() < 0.5:
        return dict(family="shortfall", L=f"3*0.{'3'*n}", R="1", label=False)                     # 3 x 0.33..3 = 1 ?
    return dict(family="shortfall", L=f"3*(10**{n}-1)/(3*10**{n})+10**(-{n})", R="1", label=True)


_VALS = ["pi", "phi", "sqrt(2)", "sqrt(3)", "log(2)", "exp(pi)", "2*pi/phi**2", "pi**2/6", "sqrt(5)", "exp(1)"]


def gen_value(rng):
    e = rng.choice(_VALS); k = rng.randint(3, 28)
    v = Decimal(str(sp.N(parse(e), 80)))
    q = v.quantize(Decimal(1).scaleb(-k), rounding=ROUND_HALF_UP)
    if rng.random() < 0.5:
        return dict(family="value", expr=e, claimed=format(q, "f"), label=True)
    bad = q + rng.choice([-1, 1]) * Decimal(1).scaleb(-k)
    return dict(family="value", expr=e, claimed=format(bad, "f"), label=False)


def gen_bound(rng):
    a = round(rng.uniform(0.5, 5), 3); x0 = round(rng.uniform(1, 9), 4)
    w = rng.choice([0.002, 0.01, 0.05]); kk = rng.choice([1, 3]); eps = rng.choice([0.001, 0.01, 0.05])
    true = rng.random() < 0.5
    c = a * (1 - eps) if true else a * (1 + eps)
    return dict(family="bound", expr=f"{a}*exp(-((x-{x0})/{w})**2)*cos({kk}*(x-{x0}))", c=format(Decimal(c).quantize(Decimal("1e-9")), "f"), label=true)


_QD = {"m": "M", "a": "L T^-2", "v": "L T^-1", "d": "L", "t": "T", "F": "M L T^-2", "En": "M L^2 T^-2", "p": "M L T^-1", "P": "M L^2 T^-3"}
_TRUE_EQ = ["F = m*a", "En = m*v**2/2", "p = m*v", "v = d/t", "P = F*v", "En = F*d", "a = v/t", "En = P*t", "d = v*t", "F = p/t"]
_BAD_EQ = ["En = m*v", "F = m*v", "p = m*v**2", "v = d*t", "P = F*d", "En = m*v**2 + m*v", "a = v*t", "En = F*t", "d = v/t", "F = m*sin(d)", "En = m*exp(t)", "P = F*v + m*a"]


def gen_dims(rng):
    true = rng.random() < 0.5
    eq = rng.choice(_TRUE_EQ if true else _BAD_EQ)
    if rng.random() < 0.5:                                        # harmless decoration: a dimensionless factor
        lhs, rhs = eq.split("="); eq = f"{lhs.strip()} = {rng.choice(['2', '3', 'pi', '1/2'])}*({rhs.strip()})"
        if not true and "sin" in eq or not true and "exp" in eq: pass
    return dict(family="dimensions", eq=eq, label=true)


def make_seed(seed):
    rng = random.Random(seed)
    out = [gen_identity(rng) for _ in range(40)] + [gen_shortfall(rng) for _ in range(10)] + [gen_value(rng) for _ in range(30)]
    out += [gen_bound(rng) for _ in range(20)] + [gen_dims(rng) for _ in range(30)]
    return out


# ------------------------------------------------------------------ baselines and engine
def fparse(s):
    return sp.parse_expr(s.replace("^", "**"), local_dict={"phi": sp.GoldenRatio, "pi": sp.pi, "euler": sp.E})


_LOOKUP = {("sin(2*x)**2+cos(2*x)**2", "1"), ("sin(x+y)", "sin(x)*cos(y)+cos(x)*sin(y)"), ("x**2-4", "(x-2)*(x+2)"),
           ("exp(2*x+y)", "exp(x)**2*exp(y)"), ("phi**4", "3*phi+2"), ("(x+2)**3", "x**3+6*x**2+12*x+8")}
_LOOKUP_VALUES = {("pi", "3.14159265358979"), ("phi", "1.6180339887"), ("sqrt(2)", "1.41421356237"), ("exp(1)", "2.71828182845")}
_LOOKUP_EQ = {"F = m*a", "En = m*v**2/2", "p = m*v", "v = d/t"}


def float_check(c, npts, rng):
    f = c["family"]
    if f in ("identity", "shortfall"):
        d = fparse(c["L"]) - fparse(c["R"])
        syms = sorted(d.free_symbols, key=str)
        fn = sp.lambdify(syms, d, "numpy")
        fl = sp.lambdify(syms, fparse(c["L"]), "numpy")
        for _ in range(npts):
            pt = [rng.uniform(0.1, 3.0) for _ in syms]
            if abs(float(fn(*pt))) > 1e-9 * max(1.0, abs(float(fl(*pt)))):
                return False
        return True
    if f == "value":
        return abs(float(sp.N(fparse(c["expr"]), 15)) - float(c["claimed"])) <= 1e-9
    if f == "bound":
        fn = sp.lambdify(sp.Symbol("x"), fparse(c["expr"]), "numpy")
        xs = np.linspace(0, 10, 400 if npts > 1 else 50)
        return bool(np.max(np.abs(fn(xs))) > float(c["c"]))
    return False                                                  # numeric checks cannot see units


def decide(method, c, eng, rng):
    f = c["family"]
    if method == "Trust":
        return True
    if method == "Lookup":
        if f in ("identity", "shortfall"): return (c["L"], c["R"]) in _LOOKUP
        if f == "value": return (c["expr"], c["claimed"]) in _LOOKUP_VALUES
        if f == "dimensions": return c["eq"] in _LOOKUP_EQ
        return False
    if method == "Float1": return float_check(c, 1, rng)
    if method == "Float5": return float_check(c, 5, rng)
    if f in ("identity", "shortfall"): return eng.identity(c["L"], c["R"]).accepted
    if f == "value": return eng.value(c["expr"], c["claimed"]).accepted
    if f == "bound": return eng.exceeds(c["expr"], "x", 0, 10, c["c"]).status == "PROVED"
    return eng.dimensions(c["eq"], _QD).accepted


def run_seed(seed):
    claims = make_seed(seed); eng = NKPMath(seed=seed); rng = random.Random(seed + 1)
    tally = {m: {fam: [0, 0, 0, 0] for fam in FAMILIES} for m in METHODS}            # correct, total, false-accepts, false-total
    for c in claims:
        for m in METHODS:
            d = decide(m, c, eng, rng)
            t = tally[m][c["family"]]
            t[0] += int(d == c["label"]); t[1] += 1
            if not c["label"]:
                t[3] += 1; t[2] += int(d)
    return tally


# ------------------------------------------------------------------ part 2: real claims from the user's documents
REAL = [
    # (kind, args, expected, source)
    ("identity", ("2*cos(pi/5)", "phi"), "accept", "screenshot:gemini-identity"),
    ("identity", ("phi", "1+1/phi"), "accept", "screenshot:gemini-identity"),
    ("identity", ("phi**2", "phi+1"), "accept", "doc:copilot-parameters"),
    ("identity", ("3*0.3333", "1"), "reject", "screenshot:triadic-flaw"),
    ("identity", ("0.3333+0.3333+0.3333", "0.9999"), "accept", "user:own-arithmetic"),
    ("identity", ("0.3333+0.3333+0.3333+0.0001", "1"), "accept", "user:own-arithmetic"),
    ("identity", ("1-3*(10**8-1)/(3*10**8)", "10**(-8)"), "accept", "screenshot:triadic-flaw"),
    ("value", ("2*pi/phi**2", "2.3999632297"), "accept", "doc:sieve-script"),
    ("value", ("3**9", "19683"), "accept", "screenshot:gemini-tictactoe"),
    ("exceeds", ("sin(x*pi)*phi**(x/10)/(pi*phi)", "x", 0, 10, "0.3333333333333333"), "reject", "doc:gate-script"),
    ("dimensions", ("Gmn = 8*pi*G/c**4*Tmn", {"Gmn": "L^-2", "G": "L^3 M^-1 T^-2", "c": "L T^-1", "Tmn": "M L^-1 T^-2"}), "accept", "gr-baseline"),
    ("dimensions", ("Gmn = 8*pi*G/c**4*(I*sqrt(X))**(1+b)*Tmn/sqrt(X)", {"Gmn": "L^-2", "G": "L^3 M^-1 T^-2", "c": "L T^-1", "Tmn": "M L^-1 T^-2", "X": "M L^-1 T^-2"}), "conditional", "screenshot:copilot-single-equation"),
    ("dimensions", ("Pw = al*Cc*f*V**2", {"Pw": "M L^2 T^-3", "Cc": "Q^2 T^2 M^-1 L^-2", "f": "T^-1", "V": "M L^2 T^-2 Q^-1"}), "accept", "screenshot:google-ai-cpu"),
    ("dimensions", ("Pj = Ic**2*Rr", {"Pj": "M L^2 T^-3", "Ic": "Q T^-1", "Rr": "M L^2 T^-1 Q^-2"}), "accept", "screenshot:google-ai-cpu"),
    ("dimensions", ("Ic = Isat*exp(V)", {"Ic": "Q T^-1", "Isat": "Q T^-1", "V": "M L^2 T^-2 Q^-1"}), "reject", "screenshot:google-ai-cpu"),
    ("dimensions", ("Gmn = rho", {"Gmn": "L^-2"}), "reject", "doc:unified-equation"),
    ("coincidence", (360 * (1 - 1 / ((1 + 5 ** 0.5) / 2)), 137.035999084, "360*(1-1/phi)"), "reject", "user:hydrogen-similarity"),
]


def run_real():
    eng = NKPMath(seed=0); ok = 0
    print(f"{'#':>2} {'source':<34}{'verdict':<13}{'expected':<12} claim / detail")
    for i, (kind, args, exp, src) in enumerate(REAL, 1):
        v = getattr(eng, kind)(*args, source=src)
        got = "accept" if v.accepted else "reject" if v.status in ("REFUTED", "ILL_FORMED", "COINCIDENCE") else "conditional"
        good = got == exp; ok += good
        name = args[0] if isinstance(args[0], str) else f"{args[0]:.4f} vs {args[1]}"
        print(f"{i:>2} {src:<34}{v.status:<13}{exp:<12} {'OK ' if good else 'BAD'} {str(name)[:46]} | {v.detail[:90]}")
    print(f"\nM4: {ok} of {len(REAL)} real claims received the expected verdict -> {'PASS' if ok == len(REAL) else 'FAIL'}")
    print("\nsource track record (true / false / open -> reliability):")
    for s, t, f, o, r in eng.ledger.table():
        print(f"  {s:<34}{t} / {f} / {o}  -> {r:.2f}")
    return ok == len(REAL)


# ------------------------------------------------------------------ reporting
def se(x):
    return statistics.stdev(x) / math.sqrt(len(x)) if len(x) > 1 else 0.0


def report(files):
    runs = []
    for fn in files: runs += json.load(open(fn))
    n = len(runs)

    def acc(r, m, fams):
        c = sum(r[m][f][0] for f in fams); t = sum(r[m][f][1] for f in fams); return c / t

    def fa(r, m):
        a = sum(r[m][f][2] for f in FAMILIES); b = sum(r[m][f][3] for f in FAMILIES); return a / b
    print(f"{n} seeds, {n*130} claims.   accuracy by family (mean over seeds)")
    print(f"{'family':<12}" + "".join(f"{m:>11}" for m in METHODS))
    for fam in FAMILIES + ["ALL"]:
        fams = FAMILIES if fam == "ALL" else [fam]
        print(f"{fam:<12}" + "".join(f"{statistics.mean(acc(r, m, fams) for r in runs):>11.3f}" for m in METHODS))
    print(f"{'false-accept':<12}" + "".join(f"{statistics.mean(fa(r, m) for r in runs):>11.3f}" for m in METHODS))
    best = lambda fams: max(METHODS[:-1], key=lambda m: statistics.mean(acc(r, m, fams) for r in runs))
    print("\nPRE-REGISTERED CRITERIA")
    m1 = statistics.mean(fa(r, "NKP-Math") for r in runs)
    print(f"  M1 false-accept rate {m1:.3f}  {'PASS' if m1 <= 0.01 else 'FAIL'}")
    b = best(FAMILIES); d = [acc(r, "NKP-Math", FAMILIES) - acc(r, b, FAMILIES) for r in runs]
    print(f"  M2 overall vs best baseline ({b}): {statistics.mean(d):+.3f}±{se(d):.3f}  {'PASS' if statistics.mean(d) - 2 * se(d) > 0 else 'FAIL'}")
    ok3 = True
    for fam in FAMILIES:
        b = best([fam]); d = [acc(r, "NKP-Math", [fam]) - acc(r, b, [fam]) for r in runs]
        good = statistics.mean(d) + 2 * se(d) >= 0; ok3 &= good
        print(f"  M3 {fam:<11} vs best baseline ({b}): {statistics.mean(d):+.3f}±{se(d):.3f}  {'ok' if good else 'WORSE'}")
    print(f"  M3 overall: {'PASS' if ok3 else 'FAIL'}")


if __name__ == "__main__":
    a = sys.argv
    if "--real" in a:
        run_real()
    elif "--report" in a:
        report(a[a.index("--report") + 1:])
    else:
        lo, hi = map(int, a[a.index("--seeds") + 1].split(":"))
        res = [run_seed(s) for s in range(lo, hi)]
        json.dump(res, open(a[a.index("--out") + 1], "w"))
        print(f"seeds {lo}:{hi} done")

ValueError: '--seeds' is not in list

In [ ]:

"""
Red-team set for NKP-Math: hand-written claims chosen to FOOL the engine (domain traps, precision limits,
hard-but-true identities, subtle falsehoods, near-integers, tricky bounds, unit traps, numerology).

Set A and set B contain DIFFERENT instances of the SAME categories.  Both were written before any fix:
fix the engine using A, then B tells you whether the fix generalizes (B is held out).
Expected verdict classes: accept | reject | conditional     (UNDETERMINED counts as "reject": no proof, no acceptance)

    python nkp_math_redteam.py          run both sets
"""
from __future__ import annotations

import sys
from decimal import ROUND_HALF_UP, Decimal, getcontext

import mpmath as mp

from nkp_math import NKPMath

getcontext().prec = 140
POS = {"x": (0.1, 3.0), "y": (0.1, 3.0)}


def digits(const: str, k: int, delta: int = 0) -> str:
    """Independent computation of a constant to k decimals (mpmath at 160 digits), optionally last digit + delta."""
    with mp.workdps(160):
        v = {"pi": mp.pi, "e": mp.e, "sqrt2": mp.sqrt(2)}[const]
        d = Decimal(mp.nstr(v, 150))
    q = d.quantize(Decimal(1).scaleb(-k), rounding=ROUND_HALF_UP) + delta * Decimal(1).scaleb(-k)
    return format(q, "f")


def dm(kind, **kw):
    return kind, kw


RED_A = [
    ("A1  sqrt(x^2) = x, no domain stated", "identity", dict(lhs="sqrt(x**2)", rhs="x"), "reject"),
    ("A2  sqrt(x^2) = x, declared x>0", "identity", dict(lhs="sqrt(x**2)", rhs="x", domain=POS), "accept"),
    ("A3  log(x^2) = 2 log x, no domain", "identity", dict(lhs="log(x**2)", rhs="2*log(x)"), "reject"),
    ("A4  log(x^2) = 2 log x, declared x>0", "identity", dict(lhs="log(x**2)", rhs="2*log(x)", domain=POS), "accept"),
    ("A5  x = x + 1e-60 x^2", "identity", dict(lhs="x", rhs="x + 10**(-60)*x**2"), "reject"),
    ("A6  pi to 55 decimals (correct)", "value", dict(expr="pi", claimed=digits("pi", 55)), "accept"),
    ("A7  pi to 55 decimals (last digit +1)", "value", dict(expr="pi", claimed=digits("pi", 55, +1)), "reject"),
    ("A8  sin3x = 3sinx - 4sin^3x", "identity", dict(lhs="sin(3*x)", rhs="3*sin(x)-4*sin(x)**3"), "accept"),
    ("A9  cos^4 - sin^4 = cos2x", "identity", dict(lhs="cos(x)**4-sin(x)**4", rhs="cos(2*x)"), "accept"),
    ("A10 cos5x = 16c^5-20c^3+5c", "identity", dict(lhs="cos(5*x)", rhs="16*cos(x)**5-20*cos(x)**3+5*cos(x)"), "accept"),
    ("A11 e^(pi sqrt163) = 640320^3+744", "identity", dict(lhs="exp(pi*sqrt(163))", rhs="640320**3+744"), "reject"),
    ("A12 e^pi - pi = 20", "identity", dict(lhs="exp(pi)-pi", rhs="20"), "reject"),
    ("A13 (x+y)^2 = x^2+y^2", "identity", dict(lhs="(x+y)**2", rhs="x**2+y**2"), "reject"),
    ("A14 sin(x+y) = sin x + sin y", "identity", dict(lhs="sin(x+y)", rhs="sin(x)+sin(y)"), "reject"),
    ("A15 sin^2 x = x^2 - x^4/3 (truncated series)", "identity", dict(lhs="sin(x)**2", rhs="x**2-x**4/3"), "reject"),
    ("A16 |sin x e^-x| > 0.35 on [0,10]  (sup .3224)", "exceeds", dict(expr="sin(x)*exp(-x)", var="x", lo=0, hi=10, c="0.35"), "reject"),
    ("A17 |sin x e^-x| > 0.30 on [0,10]", "exceeds", dict(expr="sin(x)*exp(-x)", var="x", lo=0, hi=10, c="0.30"), "accept"),
    ("A18 |sin x e^-x| > 0.3223 on [0,10]", "exceeds", dict(expr="sin(x)*exp(-x)", var="x", lo=0, hi=10, c="0.3223"), "accept"),
    ("A19 |sin x e^-x| > 0.3224 on [0,10]", "exceeds", dict(expr="sin(x)*exp(-x)", var="x", lo=0, hi=10, c="0.3224"), "reject"),
    ("A20 T = 2 pi sqrt(L/g)", "dimensions", dict(equation="T = 2*pi*sqrt(L/g)", dims={"T": "T", "L": "L", "g": "L T^-2"}), "accept"),
    ("A21 T = 2 pi sqrt(g/L)", "dimensions", dict(equation="T = 2*pi*sqrt(g/L)", dims={"T": "T", "L": "L", "g": "L T^-2"}), "reject"),
    ("A22 E = m v^n (unknown n)", "dimensions", dict(equation="En = m*v**n", dims={"En": "M L^2 T^-2", "m": "M", "v": "L T^-1"}), "conditional"),
    ("A23 E = h f", "dimensions", dict(equation="En = h*f", dims={"En": "M L^2 T^-2", "h": "M L^2 T^-1", "f": "T^-1"}), "accept"),
    ("A24 e^pi - pi ~ 20 is meaningful", "coincidence", dict(a=19.99909997, b=20.0, formula="exp(pi)-pi"), "reject"),
    ("A25 6 pi^5 ~ proton/electron mass ratio", "coincidence", dict(a=1836.118108, b=1836.15267343, formula="6*pi**5"), "reject"),
    ("A26 (x^2-1)/(x-1) = x+1", "identity", dict(lhs="(x**2-1)/(x-1)", rhs="x+1"), "accept"),
    ("A27 x/x = 1", "identity", dict(lhs="x/x", rhs="1"), "accept"),
]

RED_B = [
    ("B1  sqrt(x^4) = x^2 (true for all reals)", "identity", dict(lhs="sqrt(x**4)", rhs="x**2"), "accept"),
    ("B2  sqrt(x^2) = |x| (true for all reals)", "identity", dict(lhs="sqrt(x**2)", rhs="Abs(x)"), "accept"),
    ("B3  log(x^3) = 3 log x, no domain", "identity", dict(lhs="log(x**3)", rhs="3*log(x)"), "reject"),
    ("B4  log(x^3) = 3 log x, declared x>0", "identity", dict(lhs="log(x**3)", rhs="3*log(x)", domain=POS), "accept"),
    ("B5  y = y + 1e-70 y^3", "identity", dict(lhs="y", rhs="y + 10**(-70)*y**3"), "reject"),
    ("B6  e to 50 decimals (correct)", "value", dict(expr="exp(1)", claimed=digits("e", 50)), "accept"),
    ("B7  sqrt2 to 52 decimals (last digit -1)", "value", dict(expr="sqrt(2)", claimed=digits("sqrt2", 52, -1)), "reject"),
    ("B8  cos3x = 4c^3 - 3c", "identity", dict(lhs="cos(3*x)", rhs="4*cos(x)**3-3*cos(x)"), "accept"),
    ("B9  sin^4 - cos^4 = -cos2x", "identity", dict(lhs="sin(x)**4-cos(x)**4", rhs="-cos(2*x)"), "accept"),
    ("B10 sin5x = 16s^5-20s^3+5s", "identity", dict(lhs="sin(5*x)", rhs="16*sin(x)**5-20*sin(x)**3+5*sin(x)"), "accept"),
    ("B11 e^(pi sqrt67) = 5280^3+744", "identity", dict(lhs="exp(pi*sqrt(67))", rhs="5280**3+744"), "reject"),
    ("B12 pi^4 + pi^5 = e^6", "identity", dict(lhs="pi**4+pi**5", rhs="exp(6)"), "reject"),
    ("B13 (x+y)^3 = x^3+y^3", "identity", dict(lhs="(x+y)**3", rhs="x**3+y**3"), "reject"),
    ("B14 cos(x+y) = cos x + cos y", "identity", dict(lhs="cos(x+y)", rhs="cos(x)+cos(y)"), "reject"),
    ("B15 cos^2 x = 1 - x^2 + x^4/3 (truncated)", "identity", dict(lhs="cos(x)**2", rhs="1-x**2+x**4/3"), "reject"),
    ("B16 |cos x e^(-x/2)| > 0.99 on [0,10]", "exceeds", dict(expr="cos(x)*exp(-x/2)", var="x", lo=0, hi=10, c="0.99"), "accept"),
    ("B17 |cos x e^(-x/2)| > 1.01 on [0,10]", "exceeds", dict(expr="cos(x)*exp(-x/2)", var="x", lo=0, hi=10, c="1.01"), "reject"),
    ("B18 |x e^-x| > 0.3678 on [0,10]  (sup 1/e)", "exceeds", dict(expr="x*exp(-x)", var="x", lo=0, hi=10, c="0.3678"), "accept"),
    ("B19 |x e^-x| > 0.3680 on [0,10]", "exceeds", dict(expr="x*exp(-x)", var="x", lo=0, hi=10, c="0.3680"), "reject"),
    ("B20 v = sqrt(2 g h)", "dimensions", dict(equation="v = sqrt(2*g*h)", dims={"v": "L T^-1", "g": "L T^-2", "h": "L"}), "accept"),
    ("B21 v = sqrt(g/h)", "dimensions", dict(equation="v = sqrt(g/h)", dims={"v": "L T^-1", "g": "L T^-2", "h": "L"}), "reject"),
    ("B22 F = m v^n / t (unknown n)", "dimensions", dict(equation="F = m*v**n/t", dims={"F": "M L T^-2", "m": "M", "v": "L T^-1", "t": "T"}), "conditional"),
    ("B23 E = k x^2 / 2 (spring)", "dimensions", dict(equation="En = k*x**2/2", dims={"En": "M L^2 T^-2", "k": "M T^-2", "x": "L"}), "accept"),
    ("B24 2^10 ~ 10^3 is meaningful", "coincidence", dict(a=1024.0, b=1000.0, formula="2**10"), "reject"),
    ("B25 pi^2 ~ g = 9.80665 is meaningful", "coincidence", dict(a=9.8696044, b=9.80665, formula="pi**2"), "reject"),
    ("B26 (x^3-1)/(x-1) = x^2+x+1", "identity", dict(lhs="(x**3-1)/(x-1)", rhs="x**2+x+1"), "accept"),
    ("B27 x (1/x) = 1", "identity", dict(lhs="x*(1/x)", rhs="1"), "accept"),
]


def classify(v) -> str:
    if v.accepted:
        return "accept"
    return "conditional" if v.status == "CONDITIONAL" else "reject"


def run_set(name, items, eng):
    ok = 0
    print(f"\n=== red-team set {name} ===")
    for label, kind, kw, expected in items:
        v = getattr(eng, kind)(**kw)
        got = classify(v)
        good = got == expected
        ok += good
        print(f"{'ok ' if good else 'BAD'} {label:<46} expected {expected:<11} got {v.status:<12} {'' if good else '<- ' + v.detail[:95]}")
    print(f"set {name}: {ok} of {len(items)} correct")
    return ok, len(items)


if __name__ == "__main__":
    eng = NKPMath(seed=1)
    a = run_set("A", RED_A, eng)
    b = run_set("B", RED_B, eng)
    print(f"\nTOTAL {a[0] + b[0]} of {a[1] + b[1]}   (A = used for fixing, B = held out)")


=== red-team set A ===
BAD A1  sqrt(x^2) = x, no domain stated            expected reject      got CONDITIONAL  <- holds for positive inputs only; fails at x=-2.26723: lhs-rhs = 4.53445
ok  A2  sqrt(x^2) = x, declared x>0                expected accept      got SUPPORTED    
BAD A3  log(x^2) = 2 log x, no domain              expected reject      got CONDITIONAL  <- holds for positive inputs only; fails at x=-1.38969: lhs-rhs = (0.0 - 6.28319j)
ok  A4  log(x^2) = 2 log x, declared x>0           expected accept      got SUPPORTED    
ok  A5  x = x + 1e-60 x^2                          expected reject      got REFUTED      
ok  A6  pi to 55 decimals (correct)                expected accept      got VERIFIED     
ok  A7  pi to 55 decimals (last digit +1)          expected reject      got REFUTED      
ok  A8  sin3x = 3sinx - 4sin^3x                    expected accept      got PROVED       
ok  A9  cos^4 - sin^4 = cos2x                      expected accept      got PROVED       
ok  A10 cos

In [ ]:
import subprocess

# Run the red-team test suite directly
print("Evaluating engine against Red-Team Set A and Set B...")
result = subprocess.run(["python", "nkp_math_redteam.py"], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)

Evaluating engine against Red-Team Set A and Set B...

Errors: python3: can't open file '/content/nkp_math_redteam.py': [Errno 2] No such file or directory



In [ ]:
%%writefile nkp_math_redteam.py
"""
Red-team set for NKP-Math: hand-written claims chosen to FOOL the engine (domain traps, precision limits,
hard-but-true identities, subtle falsehoods, near-integers, tricky bounds, unit traps, numerology).

Set A and set B contain DIFFERENT instances of the SAME categories.  Both were written before any fix:
fix the engine using A, then B tells you whether the fix generalizes (B is held out).
Expected verdict classes: accept | reject | conditional     (UNDETERMINED counts as "reject": no proof, no acceptance)

    python nkp_math_redteam.py          run both sets
"""
from __future__ import annotations

import sys
from decimal import ROUND_HALF_UP, Decimal, getcontext

import mpmath as mp

from nkp_math import NKPMath

getcontext().prec = 140
POS = {"x": (0.1, 3.0), "y": (0.1, 3.0)}


def digits(const: str, k: int, delta: int = 0) -> str:
    """Independent computation of a constant to k decimals (mpmath at 160 digits), optionally last digit + delta."""
    with mp.workdps(160):
        v = {"pi": mp.pi, "e": mp.e, "sqrt2": mp.sqrt(2)}[const]
        d = Decimal(mp.nstr(v, 150))
    q = d.quantize(Decimal(1).scaleb(-k), rounding=ROUND_HALF_UP) + delta * Decimal(1).scaleb(-k)
    return format(q, "f")


def dm(kind, **kw):
    return kind, kw


RED_A = [
    ("A1  sqrt(x^2) = x, no domain stated", "identity", dict(lhs="sqrt(x**2)", rhs="x"), "reject"),
    ("A2  sqrt(x^2) = x, declared x>0", "identity", dict(lhs="sqrt(x**2)", rhs="x", domain=POS), "accept"),
    ("A3  log(x^2) = 2 log x, no domain", "identity", dict(lhs="log(x**2)", rhs="2*log(x)"), "reject"),
    ("A4  log(x^2) = 2 log x, declared x>0", "identity", dict(lhs="log(x**2)", rhs="2*log(x)", domain=POS), "accept"),
    ("A5  x = x + 1e-60 x^2", "identity", dict(lhs="x", rhs="x + 10**(-60)*x**2"), "reject"),
    ("A6  pi to 55 decimals (correct)", "value", dict(expr="pi", claimed=digits("pi", 55)), "accept"),
    ("A7  pi to 55 decimals (last digit +1)", "value", dict(expr="pi", claimed=digits("pi", 55, +1)), "reject"),
    ("A8  sin3x = 3sinx - 4sin^3x", "identity", dict(lhs="sin(3*x)", rhs="3*sin(x)-4*sin(x)**3"), "accept"),
    ("A9  cos^4 - sin^4 = cos2x", "identity", dict(lhs="cos(x)**4-sin(x)**4", rhs="cos(2*x)"), "accept"),
    ("A10 cos5x = 16c^5-20c^3+5c", "identity", dict(lhs="cos(5*x)", rhs="16*cos(x)**5-20*cos(x)**3+5*cos(x)"), "accept"),
    ("A11 e^(pi sqrt163) = 640320^3+744", "identity", dict(lhs="exp(pi*sqrt(163))", rhs="640320**3+744"), "reject"),
    ("A12 e^pi - pi = 20", "identity", dict(lhs="exp(pi)-pi", rhs="20"), "reject"),
    ("A13 (x+y)^2 = x^2+y^2", "identity", dict(lhs="(x+y)**2", rhs="x**2+y**2"), "reject"),
    ("A14 sin(x+y) = sin x + sin y", "identity", dict(lhs="sin(x+y)", rhs="sin(x)+sin(y)"), "reject"),
    ("A15 sin^2 x = x^2 - x^4/3 (truncated series)", "identity", dict(lhs="sin(x)**2", rhs="x**2-x**4/3"), "reject"),
    ("A16 |sin x e^-x| > 0.35 on [0,10]  (sup .3224)", "exceeds", dict(expr="sin(x)*exp(-x)", var="x", lo=0, hi=10, c="0.35"), "reject"),
    ("A17 |sin x e^-x| > 0.30 on [0,10]", "exceeds", dict(expr="sin(x)*exp(-x)", var="x", lo=0, hi=10, c="0.30"), "accept"),
    ("A18 |sin x e^-x| > 0.3223 on [0,10]", "exceeds", dict(expr="sin(x)*exp(-x)", var="x", lo=0, hi=10, c="0.3223"), "accept"),
    ("A19 |sin x e^-x| > 0.3224 on [0,10]", "exceeds", dict(expr="sin(x)*exp(-x)", var="x", lo=0, hi=10, c="0.3224"), "reject"),
    ("A20 T = 2 pi sqrt(L/g)", "dimensions", dict(equation="T = 2*pi*sqrt(L/g)", dims={"T": "T", "L": "L", "g": "L T^-2"}), "accept"),
    ("A21 T = 2 pi sqrt(g/L)", "dimensions", dict(equation="T = 2*pi*sqrt(g/L)", dims={"T": "T", "L": "L", "g": "L T^-2"}), "reject"),
    ("A22 E = m v^n (unknown n)", "dimensions", dict(equation="En = m*v**n", dims={"En": "M L^2 T^-2", "m": "M", "v": "L T^-1"}), "conditional"),
    ("A23 E = h f", "dimensions", dict(equation="En = h*f", dims={"En": "M L^2 T^-2", "h": "M L^2 T^-1", "f": "T^-1"}), "accept"),
    ("A24 e^pi - pi ~ 20 is meaningful", "coincidence", dict(a=19.99909997, b=20.0, formula="exp(pi)-pi"), "reject"),
    ("A25 6 pi^5 ~ proton/electron mass ratio", "coincidence", dict(a=1836.118108, b=1836.15267343, formula="6*pi**5"), "reject"),
    ("A26 (x^2-1)/(x-1) = x+1", "identity", dict(lhs="(x**2-1)/(x-1)", rhs="x+1"), "accept"),
    ("A27 x/x = 1", "identity", dict(lhs="x/x", rhs="1"), "accept"),
]

RED_B = [
    ("B1  sqrt(x^4) = x^2 (true for all reals)", "identity", dict(lhs="sqrt(x**4)", rhs="x**2"), "accept"),
    ("B2  sqrt(x^2) = |x| (true for all reals)", "identity", dict(lhs="sqrt(x**2)", rhs="Abs(x)"), "accept"),
    ("B3  log(x^3) = 3 log x, no domain", "identity", dict(lhs="log(x**3)", rhs="3*log(x)"), "reject"),
    ("B4  log(x^3) = 3 log x, declared x>0", "identity", dict(lhs="log(x**3)", rhs="3*log(x)", domain=POS), "accept"),
    ("B5  y = y + 1e-70 y^3", "identity", dict(lhs="y", rhs="y + 10**(-70)*y**3"), "reject"),
    ("B6  e to 50 decimals (correct)", "value", dict(expr="exp(1)", claimed=digits("e", 50)), "accept"),
    ("B7  sqrt2 to 52 decimals (last digit -1)", "value", dict(expr="sqrt(2)", claimed=digits("sqrt2", 52, -1)), "reject"),
    ("B8  cos3x = 4c^3 - 3c", "identity", dict(lhs="cos(3*x)", rhs="4*cos(x)**3-3*cos(x)"), "accept"),
    ("B9  sin^4 - cos^4 = -cos2x", "identity", dict(lhs="sin(x)**4-cos(x)**4", rhs="-cos(2*x)"), "accept"),
    ("B10 sin5x = 16s^5-20s^3+5s", "identity", dict(lhs="sin(5*x)", rhs="16*sin(x)**5-20*sin(x)**3+5*sin(x)"), "accept"),
    ("B11 e^(pi sqrt67) = 5280^3+744", "identity", dict(lhs="exp(pi*sqrt(67))", rhs="5280**3+744"), "reject"),
    ("B12 pi^4 + pi^5 = e^6", "identity", dict(lhs="pi**4+pi**5", rhs="exp(6)"), "reject"),
    ("B13 (x+y)^3 = x^3+y^3", "identity", dict(lhs="(x+y)**3", rhs="x**3+y**3"), "reject"),
    ("B14 cos(x+y) = cos x + cos y", "identity", dict(lhs="cos(x+y)", rhs="cos(x)+cos(y)"), "reject"),
    ("B15 cos^2 x = 1 - x^2 + x^4/3 (truncated)", "identity", dict(lhs="cos(x)**2", rhs="1-x**2+x**4/3"), "reject"),
    ("B16 |cos x e^(-x/2)| > 0.99 on [0,10]", "exceeds", dict(expr="cos(x)*exp(-x/2)", var="x", lo=0, hi=10, c="0.99"), "accept"),
    ("B17 |cos x e^(-x/2)| > 1.01 on [0,10]", "exceeds", dict(expr="cos(x)*exp(-x/2)", var="x", lo=0, hi=10, c="1.01"), "reject"),
    ("B18 |x e^-x| > 0.3678 on [0,10]  (sup 1/e)", "exceeds", dict(expr="x*exp(-x)", var="x", lo=0, hi=10, c="0.3678"), "accept"),
    ("B19 |x e^-x| > 0.3680 on [0,10]", "exceeds", dict(expr="x*exp(-x)", var="x", lo=0, hi=10, c="0.3680"), "reject"),
    ("B20 v = sqrt(2 g h)", "dimensions", dict(equation="v = sqrt(2*g*h)", dims={"v": "L T^-1", "g": "L T^-2", "h": "L"}), "accept"),
    ("B21 v = sqrt(g/h)", "dimensions", dict(equation="v = sqrt(g/h)", dims={"v": "L T^-1", "g": "L T^-2", "h": "L"}), "reject"),
    ("B22 F = m v^n / t (unknown n)", "dimensions", dict(equation="F = m*v**n/t", dims={"F": "M L T^-2", "m": "M", "v": "L T^-1", "t": "T"}), "conditional"),
    ("B23 E = k x^2 / 2 (spring)", "dimensions", dict(equation="En = k*x**2/2", dims={"En": "M L^2 T^-2", "k": "M T^-2", "x": "L"}), "accept"),
    ("B24 2^10 ~ 10^3 is meaningful", "coincidence", dict(a=1024.0, b=1000.0, formula="2**10"), "reject"),
    ("B25 pi^2 ~ g = 9.80665 is meaningful", "coincidence", dict(a=9.8696044, b=9.80665, formula="pi**2"), "reject"),
    ("B26 (x^3-1)/(x-1) = x^2+x+1", "identity", dict(lhs="(x**3-1)/(x-1)", rhs="x**2+x+1"), "accept"),
    ("B27 x (1/x) = 1", "identity", dict(lhs="x*(1/x)", rhs="1"), "accept"),
]


def classify(v) -> str:
    if v.accepted:
        return "accept"
    return "conditional" if v.status == "CONDITIONAL" else "reject"


def run_set(name, items, eng):
    ok = 0
    print(f"\n=== red-team set {name} ===")
    for label, kind, kw, expected in items:
        v = getattr(eng, kind)(**kw)
        got = classify(v)
        good = got == expected
        ok += good
        print(f"{'ok ' if good else 'BAD'} {label:<46} expected {expected:<11} got {v.status:<12} {'' if good else '<- ' + v.detail[:95]}")
    print(f"set {name}: {ok} of {len(items)} correct")
    return ok, len(items)


if __name__ == "__main__":
    eng = NKPMath(seed=1)
    a = run_set("A", RED_A, eng)
    b = run_set("B", RED_B, eng)
    print(f"\nTOTAL {a[0] + b[0]} of {a[1] + b[1]}   (A = used for fixing, B = held out)")

Writing nkp_math_redteam.py


In [ ]:
# Run the newly written red-team test suite
import subprocess
print("Evaluating engine against Red-Team Set A and Set B...")
result = subprocess.run(["python", "nkp_math_redteam.py"], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)

Evaluating engine against Red-Team Set A and Set B...

=== red-team set A ===
BAD A1  sqrt(x^2) = x, no domain stated            expected reject      got CONDITIONAL  <- holds for positive inputs only; fails at x=-2.26723: lhs-rhs = 4.53445
ok  A2  sqrt(x^2) = x, declared x>0                expected accept      got SUPPORTED    
BAD A3  log(x^2) = 2 log x, no domain              expected reject      got CONDITIONAL  <- holds for positive inputs only; fails at x=-1.38969: lhs-rhs = (0.0 - 6.28319j)
ok  A4  log(x^2) = 2 log x, declared x>0           expected accept      got SUPPORTED    
ok  A5  x = x + 1e-60 x^2                          expected reject      got REFUTED      
ok  A6  pi to 55 decimals (correct)                expected accept      got VERIFIED     
ok  A7  pi to 55 decimals (last digit +1)          expected reject      got REFUTED      
ok  A8  sin3x = 3sinx - 4sin^3x                    expected accept      got PROVED       
ok  A9  cos^4 - sin^4 = cos2x                  

In [ ]:
import pandas as pd
from nkp_math import NKPMath

# Initialize engine and process real claims using the already loaded REAL variable
eng = NKPMath(seed=0)
for kind, args, exp, src in REAL:
    getattr(eng, kind)(*args, source=src)

# Extract table data
table_data = eng.ledger.table()

# Convert to a DataFrame for clean formatting and export
df_ledger = pd.DataFrame(table_data, columns=[
    'Source',
    'True Claims',
    'False Claims',
    'Open Claims',
    'Reliability'
])

# Display the DataFrame
display(df_ledger)

# Export to CSV
csv_filename = 'source_ledger.csv'
df_ledger.to_csv(csv_filename, index=False)
print(f"\nSuccessfully exported source ledger to {csv_filename}")

,Source,True Claims,False Claims,Open Claims,Reliability
0,screenshot:gemini-identity,2,0,0,0.750000
1,user:own-arithmetic,2,0,0,0.750000
2,doc:copilot-parameters,1,0,0,0.666667
3,doc:sieve-script,1,0,0,0.666667
4,screenshot:gemini-tictactoe,1,0,0,0.666667
5,gr-baseline,1,0,0,0.666667
6,screenshot:google-ai-cpu,2,1,0,0.600000
7,screenshot:triadic-flaw,1,1,0,0.500000
8,screenshot:copilot-single-equation,0,0,1,0.500000
9,doc:gate-script,0,1,0,0.333333



Successfully exported source ledger to source_ledger.csv


In [ ]:
import subprocess

# Run the benchmarks for seeds 100 to 120
print("Running final test benchmark over seeds 100:120...")
res_bench = subprocess.run([
    "python", "nkp_math_bench.py",
    "--seeds", "100:120",
    "--out", "test_100_120.json"
], capture_output=True, text=True)
print(res_bench.stdout)
if res_bench.stderr:
    print("Errors:", res_bench.stderr)

# Report results
print("\n--- Benchmark Report ---")
res_report = subprocess.run([
    "python", "nkp_math_bench.py",
    "--report", "test_100_120.json"
], capture_output=True, text=True)
print(res_report.stdout)
if res_report.stderr:
    print("Errors:", res_report.stderr)

Running final test benchmark over seeds 100:120...

Errors: python3: can't open file '/content/nkp_math_bench.py': [Errno 2] No such file or directory


--- Benchmark Report ---

Errors: python3: can't open file '/content/nkp_math_bench.py': [Errno 2] No such file or directory



In [ ]:
%%writefile nkp_math_bench.py
"""
Benchmark for NKP-Math. Two parts, with different meanings:

PART 1  Synthetic claim zoo (labels known BY CONSTRUCTION; each seed = 130 claims)
   identities (true templates; false by coefficient/trig swaps, tiny 1e-2..1e-25 perturbations,
   and decimal "shortfall" arithmetic), decimal value claims (digit-level errors),
   existence bounds (narrow Gaussian peaks), and unit-consistency equations.
   Baselines: Trust (accept all) | Lookup (text match against a textbook list) |
              Float1 (one random point, float64, tol 1e-9) | Float5 (five points).
   This part measures CAPABILITY COVERAGE: which failure modes does each approach catch?  It is not a
   contest on a shared hard task, and the generator and engine were written by the same person.

PART 2  Real claims taken from the user's own documents (ground truth from independent calculations
   done earlier in the project).  This is the meaningful part.

PRE-REGISTERED CRITERIA (written before any run):
  M1  NKP-Math false-accept rate <= 1% overall
  M2  overall accuracy beats the best baseline by > 2 SE (paired over seeds)
  M3  in no family is NKP-Math worse than the best baseline by > 2 SE
  M4  every real case receives the expected verdict
Dev seeds 0-9; final test = fresh seeds 100-119, run once.

    python nkp_math_bench.py --real
    python nkp_math_bench.py --seeds 0:10 --out dev.json ;  python nkp_math_bench.py --report dev.json
"""
from __future__ import annotations

import json
import math
import random
import re
import statistics
import sys
from decimal import ROUND_HALF_UP, Decimal, getcontext

import mpmath as mp
import numpy as np
import sympy as sp

from nkp_math import NKPMath, parse

getcontext().prec = 90
FAMILIES = ["identity", "shortfall", "value", "bound", "dimensions"]
METHODS = ["Trust", "Lookup", "Float1", "Float5", "NKP-Math"]


def fib(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a


# ------------------------------------------------------------------ generators
def _true_identity(rng):
    k = rng.randrange(9); a = rng.randint(2, 6)
    return [
        (f"sin({a}*x)**2+cos({a}*x)**2", "1"),
        (f"(x+{a})**3", f"x**3+{3*a}*x**2+{3*a*a}*x+{a**3}"),
        (f"x**2-{a*a}", f"(x-{a})*(x+{a})"),
        (f"cos({2*a}*x)", f"1-2*sin({a}*x)**2"),
        ("sin(x+y)", "sin(x)*cos(y)+cos(x)*sin(y)"),
        (f"exp({a}*x+y)", f"exp(x)**{a}*exp(y)"),
        (f"log(x**{a}*y)", f"{a}*log(x)+log(y)"),
        (f"x**{a}-1", "(x-1)*(" + "+".join(f"x**{j}" for j in range(a)) + ")"),
        (f"phi**{a+2}", f"{fib(a+2)}*phi+{fib(a+1)}"),
    ][k]


def _oracle_identity(L, R):
    """Ground truth for mutated identities, computed independently of the engine (80 digits, 24 points)."""
    r = random.Random(7)
    d = parse(L) - parse(R)
    syms = sorted(d.free_symbols, key=str)
    f = sp.lambdify(syms, d, modules="mpmath")
    mp.mp.dps = 80
    try:
        for _ in range(24):
            pt = [mp.mpf(0.1) + mp.mpf(2.9) * mp.mpf(r.getrandbits(250)) / mp.mpf(2) ** 250 for _ in syms]
            if abs(f(*pt)) > mp.mpf(10) ** -60:
                return False
    finally:
        mp.mp.dps = 50
    return True


def gen_identity(rng):
    L, R = _true_identity(rng)
    if rng.random() < 0.5:
        return dict(family="identity", L=L, R=R, label=True)
    mode = rng.choice(["perturb", "perturb", "coef", "swap"])
    if mode == "perturb":
        R2 = f"({R})+10**(-{rng.choice([2, 4, 6, 9, 12, 16, 20, 25])})*x**2"
    elif mode == "coef":
        nums = [m for m in re.finditer(r"(?<![\w.*^])\d+(?![\w.])", R)] or [m for m in re.finditer(r"\d+", R)]
        if nums:
            m = rng.choice(nums); R2 = R[:m.start()] + str(int(m.group()) + 1) + R[m.end():]
        else:
            R2 = f"({R})+x"                                           # no numerals to alter: add a term instead
    else:
        R2 = R.replace("sin", "cos", 1) if "sin" in R else R.replace("exp", "exp(1)*exp", 1) if "exp" in R else f"({R})*2"
    return dict(family="identity", L=L, R=R2, label=_oracle_identity(L, R2))


def gen_shortfall(rng):
    n = rng.randint(3, 25)
    if rng.random() < 0.5:
        return dict(family="shortfall", L=f"3*0.{'3'*n}", R="1", label=False)                     # 3 x 0.33..3 = 1 ?
    return dict(family="shortfall", L=f"3*(10**{n}-1)/(3*10**{n})+10**(-{n})", R="1", label=True)


_VALS = ["pi", "phi", "sqrt(2)", "sqrt(3)", "log(2)", "exp(pi)", "2*pi/phi**2", "pi**2/6", "sqrt(5)", "exp(1)"]


def gen_value(rng):
    e = rng.choice(_VALS); k = rng.randint(3, 28)
    v = Decimal(str(sp.N(parse(e), 80)))
    q = v.quantize(Decimal(1).scaleb(-k), rounding=ROUND_HALF_UP)
    if rng.random() < 0.5:
        return dict(family="value", expr=e, claimed=format(q, "f"), label=True)
    bad = q + rng.choice([-1, 1]) * Decimal(1).scaleb(-k)
    return dict(family="value", expr=e, claimed=format(bad, "f"), label=False)


def gen_bound(rng):
    a = round(rng.uniform(0.5, 5), 3); x0 = round(rng.uniform(1, 9), 4)
    w = rng.choice([0.002, 0.01, 0.05]); kk = rng.choice([1, 3]); eps = rng.choice([0.001, 0.01, 0.05])
    true = rng.random() < 0.5
    c = a * (1 - eps) if true else a * (1 + eps)
    return dict(family="bound", expr=f"{a}*exp(-((x-{x0})/{w})**2)*cos({kk}*(x-{x0}))", c=format(Decimal(c).quantize(Decimal("1e-9")), "f"), label=true)


_QD = {"m": "M", "a": "L T^-2", "v": "L T^-1", "d": "L", "t": "T", "F": "M L T^-2", "En": "M L^2 T^-2", "p": "M L T^-1", "P": "M L^2 T^-3"}
_TRUE_EQ = ["F = m*a", "En = m*v**2/2", "p = m*v", "v = d/t", "P = F*v", "En = F*d", "a = v/t", "En = P*t", "d = v*t", "F = p/t"]
_BAD_EQ = ["En = m*v", "F = m*v", "p = m*v**2", "v = d*t", "P = F*d", "En = m*v**2 + m*v", "a = v*t", "En = F*t", "d = v/t", "F = m*sin(d)", "En = m*exp(t)", "P = F*v + m*a"]


def gen_dims(rng):
    true = rng.random() < 0.5
    eq = rng.choice(_TRUE_EQ if true else _BAD_EQ)
    if rng.random() < 0.5:                                        # harmless decoration: a dimensionless factor
        lhs, rhs = eq.split("="); eq = f"{lhs.strip()} = {rng.choice(['2', '3', 'pi', '1/2'])}*({rhs.strip()})"
        if not true and "sin" in eq or not true and "exp" in eq: pass
    return dict(family="dimensions", eq=eq, label=true)


def make_seed(seed):
    rng = random.Random(seed)
    out = [gen_identity(rng) for _ in range(40)] + [gen_shortfall(rng) for _ in range(10)] + [gen_value(rng) for _ in range(30)]
    out += [gen_bound(rng) for _ in range(20)] + [gen_dims(rng) for _ in range(30)]
    return out


# ------------------------------------------------------------------ baselines and engine
def fparse(s):
    return sp.parse_expr(s.replace("^", "**"), local_dict={"phi": sp.GoldenRatio, "pi": sp.pi, "euler": sp.E})


_LOOKUP = {("sin(2*x)**2+cos(2*x)**2", "1"), ("sin(x+y)", "sin(x)*cos(y)+cos(x)*sin(y)"), ("x**2-4", "(x-2)*(x+2)"),
           ("exp(2*x+y)", "exp(x)**2*exp(y)"), ("phi**4", "3*phi+2"), ("(x+2)**3", "x**3+6*x**2+12*x+8")}
_LOOKUP_VALUES = {("pi", "3.14159265358979"), ("phi", "1.6180339887"), ("sqrt(2)", "1.41421356237"), ("exp(1)", "2.71828182845")}
_LOOKUP_EQ = {"F = m*a", "En = m*v**2/2", "p = m*v", "v = d/t"}


def float_check(c, npts, rng):
    f = c["family"]
    if f in ("identity", "shortfall"):
        d = fparse(c["L"]) - fparse(c["R"])
        syms = sorted(d.free_symbols, key=str)
        fn = sp.lambdify(syms, d, "numpy")
        fl = sp.lambdify(syms, fparse(c["L"]), "numpy")
        for _ in range(npts):
            pt = [rng.uniform(0.1, 3.0) for _ in syms]
            if abs(float(fn(*pt))) > 1e-9 * max(1.0, abs(float(fl(*pt)))):
                return False
        return True
    if f == "value":
        return abs(float(sp.N(fparse(c["expr"]), 15)) - float(c["claimed"])) <= 1e-9
    if f == "bound":
        fn = sp.lambdify(sp.Symbol("x"), fparse(c["expr"]), "numpy")
        xs = np.linspace(0, 10, 400 if npts > 1 else 50)
        return bool(np.max(np.abs(fn(xs))) > float(c["c"]))
    return False                                                  # numeric checks cannot see units


def decide(method, c, eng, rng):
    f = c["family"]
    if method == "Trust":
        return True
    if method == "Lookup":
        if f in ("identity", "shortfall"): return (c["L"], c["R"]) in _LOOKUP
        if f == "value": return (c["expr"], c["claimed"]) in _LOOKUP_VALUES
        if f == "dimensions": return c["eq"] in _LOOKUP_EQ
        return False
    if method == "Float1": return float_check(c, 1, rng)
    if method == "Float5": return float_check(c, 5, rng)
    if f in ("identity", "shortfall"): return eng.identity(c["L"], c["R"]).accepted
    if f == "value": return eng.value(c["expr"], c["claimed"]).accepted
    if f == "bound": return eng.exceeds(c["expr"], "x", 0, 10, c["c"]).status == "PROVED"
    return eng.dimensions(c["eq"], _QD).accepted


def run_seed(seed):
    claims = make_seed(seed); eng = NKPMath(seed=seed); rng = random.Random(seed + 1)
    tally = {m: {fam: [0, 0, 0, 0] for fam in FAMILIES} for m in METHODS}            # correct, total, false-accepts, false-total
    for c in claims:
        for m in METHODS:
            d = decide(m, c, eng, rng)
            t = tally[m][c["family"]]
            t[0] += int(d == c["label"]); t[1] += 1
            if not c["label"]:
                t[3] += 1; t[2] += int(d)
    return tally


# ------------------------------------------------------------------ part 2: real claims from the user's documents
REAL = [
    # (kind, args, expected, source)
    ("identity", ("2*cos(pi/5)", "phi"), "accept", "screenshot:gemini-identity"),
    ("identity", ("phi", "1+1/phi"), "accept", "screenshot:gemini-identity"),
    ("identity", ("phi**2", "phi+1"), "accept", "doc:copilot-parameters"),
    ("identity", ("3*0.3333", "1"), "reject", "screenshot:triadic-flaw"),
    ("identity", ("0.3333+0.3333+0.3333", "0.9999"), "accept", "user:own-arithmetic"),
    ("identity", ("0.3333+0.3333+0.3333+0.0001", "1"), "accept", "user:own-arithmetic"),
    ("identity", ("1-3*(10**8-1)/(3*10**8)", "10**(-8)"), "accept", "screenshot:triadic-flaw"),
    ("value", ("2*pi/phi**2", "2.3999632297"), "accept", "doc:sieve-script"),
    ("value", ("3**9", "19683"), "accept", "screenshot:gemini-tictactoe"),
    ("exceeds", ("sin(x*pi)*phi**(x/10)/(pi*phi)", "x", 0, 10, "0.3333333333333333"), "reject", "doc:gate-script"),
    ("dimensions", ("Gmn = 8*pi*G/c**4*Tmn", {"Gmn": "L^-2", "G": "L^3 M^-1 T^-2", "c": "L T^-1", "Tmn": "M L^-1 T^-2"}), "accept", "gr-baseline"),
    ("dimensions", ("Gmn = 8*pi*G/c**4*(I*sqrt(X))**(1+b)*Tmn/sqrt(X)", {"Gmn": "L^-2", "G": "L^3 M^-1 T^-2", "c": "L T^-1", "Tmn": "M L^-1 T^-2", "X": "M L^-1 T^-2"}), "conditional", "screenshot:copilot-single-equation"),
    ("dimensions", ("Pw = al*Cc*f*V**2", {"Pw": "M L^2 T^-3", "Cc": "Q^2 T^2 M^-1 L^-2", "f": "T^-1", "V": "M L^2 T^-2 Q^-1"}), "accept", "screenshot:google-ai-cpu"),
    ("dimensions", ("Pj = Ic**2*Rr", {"Pj": "M L^2 T^-3", "Ic": "Q T^-1", "Rr": "M L^2 T^-1 Q^-2"}), "accept", "screenshot:google-ai-cpu"),
    ("dimensions", ("Ic = Isat*exp(V)", {"Ic": "Q T^-1", "Isat": "Q T^-1", "V": "M L^2 T^-2 Q^-1"}), "reject", "screenshot:google-ai-cpu"),
    ("dimensions", ("Gmn = rho", {"Gmn": "L^-2"}), "reject", "doc:unified-equation"),
    ("coincidence", (360 * (1 - 1 / ((1 + 5 ** 0.5) / 2)), 137.035999084, "360*(1-1/phi)"), "reject", "user:hydrogen-similarity"),
]


def run_real():
    eng = NKPMath(seed=0); ok = 0
    print(f"{'#':>2} {'source':<34}{'verdict':<13}{'expected':<12} claim / detail")
    for i, (kind, args, exp, src) in enumerate(REAL, 1):
        v = getattr(eng, kind)(*args, source=src)
        got = "accept" if v.accepted else "reject" if v.status in ("REFUTED", "ILL_FORMED", "COINCIDENCE") else "conditional"
        good = got == exp; ok += good
        name = args[0] if isinstance(args[0], str) else f"{args[0]:.4f} vs {args[1]}"
        print(f"{i:>2} {src:<34}{v.status:<13}{exp:<12} {'OK ' if good else 'BAD'} {str(name)[:46]} | {v.detail[:90]}")
    print(f"\nM4: {ok} of {len(REAL)} real claims received the expected verdict -> {'PASS' if ok == len(REAL) else 'FAIL'}")
    print("\nsource track record (true / false / open -> reliability):")
    for s, t, f, o, r in eng.ledger.table():
        print(f"  {s:<34}{t} / {f} / {o}  -> {r:.2f}")
    return ok == len(REAL)


# ------------------------------------------------------------------ reporting
def se(x):
    return statistics.stdev(x) / math.sqrt(len(x)) if len(x) > 1 else 0.0


def report(files):
    runs = []
    for fn in files: runs += json.load(open(fn))
    n = len(runs)

    def acc(r, m, fams):
        c = sum(r[m][f][0] for f in fams); t = sum(r[m][f][1] for f in fams); return c / t

    def fa(r, m):
        a = sum(r[m][f][2] for f in FAMILIES); b = sum(r[m][f][3] for f in FAMILIES); return a / b
    print(f"{n} seeds, {n*130} claims.   accuracy by family (mean over seeds)")
    print(f"{'family':<12}" + "".join(f"{m:>11}" for m in METHODS))
    for fam in FAMILIES + ["ALL"]:
        fams = FAMILIES if fam == "ALL" else [fam]
        print(f"{'fam':<12}" + "".join(f"{statistics.mean(acc(r, m, fams) for r in runs):>11.3f}" for m in METHODS))
    print(f"{'false-accept':<12}" + "".join(f"{statistics.mean(fa(r, m) for r in runs):>11.3f}" for m in METHODS))
    best = lambda fams: max(METHODS[:-1], key=lambda m: statistics.mean(acc(r, m, fams) for r in runs))
    print("\nPRE-REGISTERED CRITERIA")
    m1 = statistics.mean(fa(r, "NKP-Math") for r in runs)
    print(f"  M1 false-accept rate {m1:.3f}  {'PASS' if m1 <= 0.01 else 'FAIL'}")
    b = best(FAMILIES); d = [acc(r, "NKP-Math", FAMILIES) - acc(r, b, FAMILIES) for r in runs]
    print(f"  M2 overall vs best baseline ({b}): {statistics.mean(d):+.3f}±{se(d):.3f}  {'PASS' if statistics.mean(d) - 2 * se(d) > 0 else 'FAIL'}")
    ok3 = True
    for fam in FAMILIES:
        b = best([fam]); d = [acc(r, "NKP-Math", [fam]) - acc(r, b, [fam]) for r in runs]
        good = statistics.mean(d) + 2 * se(d) >= 0; ok3 &= good
        print(f"  M3 {fam:<11} vs best baseline ({b}): {statistics.mean(d):+.3f}±{se(d):.3f}  {'ok' if good else 'WORSE'}")
    print(f"  M3 overall: {'PASS' if ok3 else 'FAIL'}")


if __name__ == "__main__":
    a = sys.argv
    if "--real" in a:
        run_real()
    elif "--report" in a:
        report(a[a.index("--report") + 1:])
    else:
        lo, hi = map(int, a[a.index("--seeds") + 1].split(":"))
        res = [run_seed(s) for s in range(lo, hi)]
        json.dump(res, open(a[a.index("--out") + 1], "w"))
        print(f"seeds {lo}:{hi} done")

Writing nkp_math_bench.py


In [ ]:
import subprocess

# Re-run the final test benchmark seeds 100:120
print("Re-running benchmark on test seeds 100:120...")
res_bench = subprocess.run([
    "python", "nkp_math_bench.py",
    "--seeds", "100:120",
    "--out", "test_100_120.json"
], capture_output=True, text=True)
print(res_bench.stdout)
if res_bench.stderr:
    print("Errors:", res_bench.stderr)

# Print report
print("\n--- Final Benchmark Report ---")
res_report = subprocess.run([
    "python", "nkp_math_bench.py",
    "--report", "test_100_120.json"
], capture_output=True, text=True)
print(res_report.stdout)
if res_report.stderr:
    print("Errors:", res_report.stderr)

Re-running benchmark on test seeds 100:120...
seeds 100:120 done


--- Final Benchmark Report ---
20 seeds, 2600 claims.   accuracy by family (mean over seeds)
family            Trust     Lookup     Float1     Float5   NKP-Math
fam               0.499      0.616      0.861      0.875      0.949
fam               0.520      0.480      0.625      0.625      1.000
fam               0.503      0.502      0.525      0.525      1.000
fam               0.492      0.507      0.517      0.608      1.000
fam               0.500      0.597      0.500      0.500      1.000
fam               0.501      0.558      0.629      0.647      0.984
false-accept      1.000      0.002      0.311      0.303      0.000

PRE-REGISTERED CRITERIA
  M1 false-accept rate 0.000  PASS
  M2 overall vs best baseline (Float5): +0.337±0.011  PASS
  M3 identity    vs best baseline (Float5): +0.074±0.017  ok
  M3 shortfall   vs best baseline (Float1): +0.375±0.039  ok
  M3 value       vs best baseline (Float1): +0.475±0.02